# Notebook 04:
# Risk Forecasting — DAG-Constrained Pipeline, Prefix-Based Feature Gating, and Causal vs Baseline Comparison

---
<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px; margin-bottom: 15px;">
<strong>📘 PURPOSE:</strong> A reproducible modeling notebook that:

<div style="margin-left: 24px; margin-top: 8px;">

**(1)** implements prefix-based feature gating to restrict the Risk forecast stage to DAG-permitted parents only (VOL, MACRO, REGIME),

**(2)** trains a DAG-constrained XGBoost model using the identical hyperparameters and expanding-window walk-forward protocol established in Notebook 03,

**(3)** computes per-ticker and aggregate RMSE and MAE deltas against the unconstrained EWMA and XGBoost baselines,

**(4)** extracts constrained feature importance to confirm that no DAG-excluded features (MOM, VAL, ML, SENT) appear, and

**(5)** computes factor-exposure entropy diagnostics to evaluate whether the constraint layer produces more balanced feature loadings.

</div>

Notebook 04 is the primary experiment — testing <strong>Hypothesis H1 (forecast accuracy)</strong> and generates the interpretability evidence for <strong>Hypothesis H4 (factor-exposure entropy)</strong>.
</div>


## CAPSTONE CONTEXT

<div style="border-left: 4px solid #6a1b9a; padding-left: 12px; margin: 10px 0;">

**Title:**  
<span style="color: purple;"><strong>Causal-Aware, Machine-Learning-Driven Risk Forecasting and Factor Construction:</strong></span> A Python–Azure Pipeline Integrating NLP, Directed Factor Constraints, and Portfolio Analytics

**Thesis:**  
A small, theory-driven manually constrained <span style="color: purple;"><strong>Directed Acyclic Graph (DAG)</strong></span> that restricts information flow can improve the stability and interpretability of ML-based risk forecasting and factor-based portfolio allocation, relative to unconstrained baselines, under regime variation and estimation noise.

**Research Question:**  
How does imposing <span style="color: purple;"><strong>manual causal constraints</strong></span> on an ML-driven risk forecasting pipeline affect forecast accuracy, portfolio performance, and interpretability compared to unconstrained baselines?

</div>

---

## MANUAL DAG (Conceptual Constraint Layer)

<div style="border: 2px solid #6a1b9a; background-color: #f3e5f5; padding: 12px; border-radius: 5px;">
<strong>🔗 DAG STRUCTURE:</strong>

```
Sentiment  →  Momentum  →  Returns
```
<span style="color: purple;">Path 1: Sentiment <strong>AMPLIFIES</strong> Momentum; Momentum <strong>FORECASTS</strong> Returns</span>

```
Value (HML)            →  Returns
```
<span style="color: purple;">Path 2: Value (HML) <strong>EXPLAINS</strong> Returns</span>

```
Volatility →  Risk     →  Allocation
```
<span style="color: purple;">Path 3: Volatility <strong>ESTIMATES</strong> Risk; Risk <strong>CONSTRAINS</strong> Allocation</span>

<span style="color: purple;"><strong>Notebook 04 Role in the DAG Pipeline:</strong></span> Notebook 03 built <strong>unconstrained</strong> baseline models that accessed <strong>all</strong> 151 non-sentiment features regardless of DAG-node membership. Notebook 04 now <strong>enforces</strong> the DAG constraint layer — restricting the Risk forecast stage to features from the <strong>Volatility</strong> node (plus exogenous MACRO and REGIME conditioning variables) and excluding Momentum, Value, ML latent, and Sentiment features entirely. The constrained model receives <strong>74 features</strong> versus 151 in the unconstrained baseline, a 51% feature reduction that serves as implicit regularization. By comparing constrained-model performance against the Notebook 03 baselines, Notebook 04 isolates the effect of DAG-based information-flow restriction on forecast accuracy and feature-importance interpretability.
</div>

---

## DAG CONSTRAINT IMPLEMENTATION — PREFIX-BASED FEATURE GATING

<div style="border: 2px solid #6a1b9a; background-color: #f3e5f5; padding: 12px; border-radius: 5px;">
<strong>🔗 ALLOWED-PARENT SET FOR THE RISK FORECAST STAGE:</strong>

The Risk node in the DAG receives information only from the Volatility node. MACRO and REGIME serve as exogenous conditioning variables available to all nodes. The feature-gating logic parses the double-underscore prefix of each column name and admits only columns matching the allowed-parent set.

| Status | Prefix | Feature Count | DAG Justification |
|:-------|:-------|:--------------|:------------------|
| <span style="color: darkgreen;"><strong>✓ ALLOWED</strong></span> | `VOL__` | 70 | Volatility → Risk (direct parent edge) |
| <span style="color: darkgreen;"><strong>✓ ALLOWED</strong></span> | `MACRO__` | 3 | Exogenous conditioning (VIX, T10Y2Y, DTB3) |
| <span style="color: darkgreen;"><strong>✓ ALLOWED</strong></span> | `REGIME__` | 1 | Exogenous conditioning (VIX high/low regime indicator) |
| <span style="color: darkred;"><strong>✗ EXCLUDED</strong></span> | `MOM__` | 56 | Momentum → Returns, not Risk |
| <span style="color: darkred;"><strong>✗ EXCLUDED</strong></span> | `VAL__` | 18 | Value → Returns, not Risk |
| <span style="color: darkred;"><strong>✗ EXCLUDED</strong></span> | `ML__` | 3 | PCA latent factors not assigned as Risk parents |
| <span style="color: darkred;"><strong>✗ EXCLUDED</strong></span> | `SENT__` | 1 | 100% NaN placeholder; Sentiment → Momentum, not Risk |
| | **TOTAL ALLOWED** | **74** | |
| | **TOTAL EXCLUDED** | **77 non-NaN + 1 NaN** | |

<span style="color: purple;"><strong>Forbidden Shortcuts:</strong></span> The DAG forbids the following information-flow paths into the Risk node: Sentiment → Risk (sentiment must flow through Momentum first), Momentum → Risk (momentum affects Returns, not Risk directly), Value → Risk (value affects Returns, not Risk directly), ML latent factors → Risk (PCA scores are general-purpose and the DAG does not assign ML factors as Risk parents). Only the Volatility → Risk edge is permitted, plus exogenous MACRO/REGIME conditioning.
</div>

---

## HYPOTHESES TESTED IN NOTEBOOK 04

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px;">
<strong>📐 HYPOTHESIS H1 — FORECAST ACCURACY:</strong>

The DAG-constrained pipeline reduces out-of-sample MAE and RMSE on 20-trading-day forward realized volatility relative to the unconstrained XGBoost baseline from Notebook 03.

**Test mechanism:** Compare per-ticker and aggregate RMSE and MAE between the constrained model (74 features) and the unconstrained XGBoost baseline (151 features), using identical hyperparameters and the identical expanding-window walk-forward protocol.
</div>

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📐 HYPOTHESIS H4 — INTERPRETABILITY (FACTOR-EXPOSURE ENTROPY):</strong>

Factor-exposure entropy increases under DAG constraints, indicating more balanced feature loadings and reduced concentration on a few cross-node shortcuts.

**Test mechanism:** Compute Shannon entropy $H = -\sum_k p_k \ln(p_k)$ over normalized gain shares from the constrained versus unconstrained XGBoost feature-importance vectors. Higher entropy indicates more dispersed feature usage.

$$
H = -\sum_{k=1}^{K} p_k \ln(p_k), \quad p_k = \frac{\text{gain}_k}{\sum_{j=1}^{K} \text{gain}_j}
$$
</div>

---

## FROZEN HYPERPARAMETERS (FROM NOTEBOOK 03 TUNING)

<div style="border: 2px solid #ff9933; background-color: #fff3cd; padding: 12px; border-radius: 5px;">
<strong>⚠️ CRITICAL FAIRNESS CONSTRAINT:</strong> Notebook 04 reuses the <strong>exact</strong> hyperparameters selected during Notebook 03 walk-forward validation. Freezing hyperparameters ensures that any performance difference between the constrained and unconstrained models is attributable to the feature restriction, not to hyperparameter variation.

| Parameter | Frozen Value |
|:----------|:-------------|
| `n_estimators` | 400 |
| `max_depth` | 4 |
| `learning_rate` | 0.05 |
| `subsample` | 0.80 |
| `colsample_bytree` | 0.80 |
| `reg_lambda` | 1.0 |
| `objective` | `reg:squarederror` |
| `random_state` | 692 |
| `tree_method` | `hist` |
</div>

---

## EVALUATION PROTOCOL (REUSED FROM NOTEBOOK 03)

<div style="border: 2px solid darkblue; background-color: #e8f4fc; padding: 12px; border-radius: 5px;">
<strong>📐 DEFINITION — Expanding-Window Walk-Forward Evaluation:</strong>

Notebook 04 reuses the identical evaluation protocol from Notebook 03 to ensure a fair comparison:

| Dimension | Value |
|:----------|:------|
| **Train–Validation–Test split** | 60%–20%–20% of the effective modeling window (time-ordered, no shuffling) |
| **Forecast horizon** | 20 trading days |
| **Walk-forward step size** | 20 trading days |
| **Label-safe boundary** | Training data truncated at `block_start − 20` to prevent label leakage from overlapping forward windows |
| **Evaluation metrics** | RMSE (primary) and MAE (robust complement), per-ticker and aggregate equal-weight mean |
| **Baseline references** | EWMA (span-63) and unconstrained XGBoost (151 features), both from Notebook 03 saved artifacts |

<span style="color: darkblue;"><strong>Walk-forward protocol:</strong></span> At each evaluation step, the training window expands by 20 trading days. A fresh XGBoost model is fit on the expanded training window and produces a forecast for the next 20-day block. The expanding-window design mimics the information set available to a portfolio manager making monthly allocation decisions.
</div>

<div style="border: 2px solid darkred; background-color: #f2dede; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🚨 LEAKAGE WARNING — TEMPORAL INTEGRITY:</strong> The walk-forward evaluation must never allow information from future dates to enter the training set. At each evaluation step, the model accesses only features and targets from dates strictly before the block start minus the 20-day forecast horizon. The label-safe boundary ensures that no partially overlapping forward-volatility targets contaminate the training window. The TARGET__-prefixed columns are loaded from a <strong>separate file</strong> (<code>target_fwd_vol.parquet</code>) and must never appear in the feature matrix.
</div>

---

## NOTEBOOK 03 BASELINE BENCHMARKS (REFERENCE)

<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px;">
<strong>📘 BASELINE PERFORMANCE (from Notebook 03 saved artifacts):</strong>

Notebook 04 loads the following Notebook 03 artifacts for direct comparison:

| Artifact | Path | Contents |
|:---------|:-----|:---------|
| Baseline metrics | `reports/tables/notebook03_baseline_rmse_mae.csv` | Per-ticker RMSE and MAE for EWMA and XGBoost |
| Baseline predictions | `reports/tables/notebook03_baseline_predictions_long.csv` | Long-form predictions for all 14 tickers and all test blocks |
| Feature importance | `reports/tables/notebook03_xgb_feature_importance.csv` | Gain-based feature importance from the unconstrained SPY model |
| Tuning results | `reports/tables/notebook03_xgb_tuning_results.csv` | 6-candidate tuning grid with validation RMSE |

**Aggregate baseline metrics (equal-weight mean across 14 tickers):**

| Model | RMSE | MAE |
|:------|:-----|:----|
| EWMA (span-63) | 0.0708 | 0.0483 |
| XGBoost (unconstrained, 151 features) | 0.0698 | 0.0491 |

<span style="color: darkorange;"><strong>⚠ Key finding from Notebook 03:</strong></span> A momentum feature (`MOM__XLF__cum_ret__10d`) appeared at rank 2 in the unconstrained SPY feature-importance hierarchy — a cross-node shortcut from Momentum into the Risk forecast. Under DAG constraints, momentum features are excluded from the Risk node's allowed-parent set. Whether the constrained model maintains or improves RMSE without momentum cross-contamination is the central empirical question of Notebook 04.
</div>

---

## INPUT AND OUTPUT FILES

<div style="border: 2px solid teal; background-color: #e0f7fa; padding: 12px; border-radius: 5px;">
<strong>📁 INPUT (from Notebooks 02 and 03):</strong>

- `data/processed/features.parquet` — Feature matrix: 2,798 rows × 152 columns (DAG-node-prefixed)
- `data/processed/target_fwd_vol.parquet` — Forward realized volatility target: 2,798 rows × 14 columns
- `data/processed/etf_returns.parquet` — Daily log returns for 14 ETFs
- `reports/tables/notebook03_baseline_rmse_mae.csv` — EWMA and XGBoost per-ticker metrics
- `reports/tables/notebook03_baseline_predictions_long.csv` — Long-form baseline predictions (all tickers, all test blocks)
- `reports/tables/notebook03_xgb_feature_importance.csv` — Unconstrained feature importance (SPY)
- `reports/tables/notebook03_xgb_tuning_results.csv` — Hyperparameter tuning grid

</div>

<div style="border: 2px solid teal; background-color: #e0f7fa; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>📁 OUTPUT (produced by Notebook 04):</strong>

**Tables (`reports/tables/`):**
- `notebook04_causal_rmse_mae.csv` — Per-ticker RMSE and MAE for the constrained model
- `notebook04_causal_predictions_long.csv` — Long-form constrained predictions (all tickers, all test blocks)
- `notebook04_dag_gating_summary.csv` — Feature-level allowed/excluded status with DAG prefix
- `notebook04_effective_window_summary.csv` — Effective modeling window confirmation
- `notebook04_split_summary.csv` — Train–validation–test split boundaries
- `causal_vs_baseline_rmse_mae.csv` — Merged comparison table with per-ticker RMSE/MAE deltas
- `notebook04_causal_feature_importance.csv` — Gain-based feature importance for representative tickers
- `notebook04_entropy_comparison.csv` — Constrained vs unconstrained entropy diagnostics
- `notebook04_prefix_gain_share.csv` — DAG-prefix-level gain share comparison
- `notebook04_top_feature_comparison.csv` — Top-10 features for constrained vs unconstrained
- `notebook04_asset_class_comparison.csv` — Asset-class-level RMSE/MAE delta summary
- `notebook04_artifact_manifest.csv` — Registry of all Notebook 04 output artifacts

**Figures (`reports/figures/`):**
- `notebook04_dag_figure.png` — Manual DAG architecture diagram with node boxes and directed edges
- `notebook04_forecast_vs_realized_SPY.png` — Forecast overlay: EWMA vs XGBoost vs Causal XGBoost vs realized
- `notebook04_causal_vs_baseline_rmse_delta.png` — RMSE delta bar chart (constrained minus baselines, per ticker)
- `notebook04_entropy_comparison.png` — Normalized entropy bar chart (constrained vs unconstrained)
- `notebook04_causal_feature_importance_SPY.png` — Top constrained features by gain (SPY)
- `notebook04_prefix_gain_share_SPY.png` — DAG-prefix gain share comparison (SPY)

**Data (`data/processed/`):**
- `notebook04_causal_predictions_long.csv` — Predictions copy for downstream Notebook 05 and Notebook 06 consumption

**Models (`reports/models/`):**
- `notebook04_xgb_causal_{TICKER}.json` — Serialized constrained XGBoost models for representative tickers

</div>

---

## FEATURE NAMING CONVENTION (REFERENCE)

<div style="border: 2px solid #007acc; background-color: #e6f0ff; padding: 12px; border-radius: 5px;">
<strong>📘 NAMING PATTERNS (inherited from Notebook 02):</strong>

**Asset-level features** (one column per ETF per window):
<code>{DAG_NODE}__{TICKER}__{feature_name}__{window}</code>

**Market-level features** (one column shared across all ETFs):
<code>{DAG_NODE}__{feature_name}__{window(optional)}</code>

**Examples:**
- `VOL__SPY__rvol__21d` — Volatility node, SPY, 21-day realized volatility (**ALLOWED** for Risk stage)
- `VOL__SPY__ewma_vol__span63` — Volatility node, SPY, EWMA volatility with span 63 (**ALLOWED** for Risk stage)
- `MACRO__vixcls` — Macro conditioning, VIX closing level (**ALLOWED** for Risk stage)
- `REGIME__vix_high` — Regime conditioning, binary high-VIX indicator (**ALLOWED** for Risk stage)
- `MOM__SPY__cum_ret__21d` — Momentum node, SPY, 21-day cumulative return (**EXCLUDED** from Risk stage)
- `VAL__SPY__hml_beta__63d` — Value node, SPY, 63-day rolling beta to HML (**EXCLUDED** from Risk stage)
- `ML__pc1` — ML latent factor, first principal component (**EXCLUDED** from Risk stage)
- `SENT__sentiment_market` — Sentiment node, daily market sentiment score (**EXCLUDED**; 100% NaN placeholder)

The double-underscore delimiter enables programmatic DAG-node parsing. The first token is always the DAG-node prefix. Notebook 04 uses the prefix to enforce causal constraints via the `gate_features_by_prefix()` function.
</div>

---

## REPRODUCIBILITY

<div style="border: 2px solid #3c763d; background-color: #dff0d8; padding: 12px; border-radius: 5px;">
<strong>✅ REPRODUCIBILITY GUARANTEES:</strong>

| Dimension | Policy |
|:----------|:-------|
| **Random Seed** | 692 (set in setup cell; passed to XGBoost `random_state`, NumPy, and Python `random`) |
| **Time Ordering** | Strictly preserved — expanding-window walk-forward with no shuffling |
| **Look-Ahead Leakage** | At each evaluation step, the model accesses only features and targets from dates before the label-safe boundary |
| **Hyperparameter Fairness** | Frozen from Notebook 03 tuning — no re-tuning on constrained features |
| **DAG Constraint Proof** | Explicit gating validation cell prints allowed and excluded feature lists with prefix counts |
| **Effective Modeling Mask** | Reuses the identical completeness mask from Notebook 03 (all features + target simultaneously non-NaN) |
| **Sentiment Exclusion** | `SENT__sentiment_market` column (100% NaN) excluded before feature gating |
| **Target-Column Governance** | All target access routes through `TARGET_COL_MAP` dictionary — bare ticker access is forbidden |
| **Model Serialization** | Constrained XGBoost models saved to JSON for reproducibility and downstream comparison |
| **Artifact Persistence** | All tables, figures, models, and predictions saved to `reports/` and `data/processed/` |
</div>

<div style="border: 2px solid darkred; background-color: #f2dede; padding: 12px; border-radius: 5px; margin-top: 10px;">
<strong>🚨 LEAKAGE WARNING:</strong> Notebook 04 trains DAG-constrained models and evaluates forecast accuracy against Notebook 03 baselines. Three leakage boundaries must hold simultaneously: (1) <strong>Temporal leakage:</strong> the expanding-window protocol with label-safe boundary prevents future target values from entering the training set. (2) <strong>Feature leakage:</strong> TARGET__-prefixed columns are loaded from a separate file (<code>target_fwd_vol.parquet</code>) and must never appear in the feature matrix. (3) <strong>DAG leakage:</strong> the prefix-gating function must exclude all MOM__, VAL__, ML__, and SENT__ columns from the constrained model's input — the gating validation cell confirms that no forbidden-prefix columns survived the filter.
</div>

---

## NOTEBOOK STRUCTURE OVERVIEW

| Cell Block | Code Block | Purpose | Key Actions |
|:-----------|:-----------|:--------|:------------|
| **SETUP** | CODE_BLOCK_A | Environment initialization | Imports, paths, seed, display options, helper functions, Notebook 03 artifact paths |
| **LOAD** | CODE_BLOCK_B | Load features, targets, returns, and Notebook 03 artifacts | Read Parquet files, load optional NB03 CSVs, validate index alignment, drop sentiment |
| **EFFECTIVE WINDOW** | CODE_BLOCK_C | Apply effective modeling mask | Completeness intersection, target-column governance map, effective window summary |
| **DAG GATING** | CODE_BLOCK_D | Define and apply DAG constraint layer | Manual DAG edge list, prefix gating function, allowed/excluded validation, gating summary |
| **SPLIT** | CODE_BLOCK_E | Time-ordered train–validation–test split | 60/20/20 split, expanding-window block indices, label-safe boundary function |
| **CAUSAL MODEL** | CODE_BLOCK_F | Train and evaluate DAG-constrained XGBoost | Frozen hyperparameters, walk-forward prediction loop, per-ticker RMSE/MAE, prediction export |
| **COMPARISON** | CODE_BLOCK_G | Merge constrained metrics with Notebook 03 baselines | RMSE/MAE delta computation, win counts versus XGBoost and EWMA baselines |
| **IMPORTANCE** | CODE_BLOCK_H | Extract constrained feature importance | Gain-based importance for representative tickers, forbidden-prefix purity check |
| **ENTROPY** | CODE_BLOCK_I | Compute factor-exposure entropy diagnostics | Shannon entropy, normalized entropy, concentration statistics, constrained vs unconstrained |
| **PLOT DATA** | CODE_BLOCK_J | Prepare data for final figures | Combined prediction table, prefix gain shares, top-feature comparison table |
| **PLOT PREP** | CODE_BLOCK_K | Build pivot tables and delta tables for plotting | Forecast pivot, RMSE/MAE delta tables, aggregate summary |
| **ASSET CLASS** | CODE_BLOCK_L | Asset-class-level decomposition | Map tickers to asset classes, compute mean RMSE/MAE deltas by asset class |
| **REGISTRY** | CODE_BLOCK_M | Preliminary artifact registry | List all expected output paths, check existence flags before figure creation |
| **FIGURES** | CODE_BLOCK_N | Generate all Notebook 04 figures and final manifest | DAG diagram, forecast overlay, RMSE delta bars, entropy bars, importance bars, prefix gain bars, artifact manifest |

---

*Notebook 04 of 7 | MScFE 692 Capstone | Steven Archuleta | WorldQuant University*


In [ ]:
# ============
# CODE_BLOCK_A
# ============

# =================================================================
# IMPORT WARNINGS FOR CLEAN NOTEBOOK OUTPUT VIA WARNING SUPPRESSION
# =================================================================

import warnings

# ============================================================
# SUPPRESS NON-CRITICAL WARNINGS TO KEEP AUDIT OUTPUT READABLE
# ============================================================

warnings.filterwarnings("ignore")

# =================================================================
# IMPORT RANDOM FOR REPRODUCIBLE PYTHON-LEVEL STOCHASTIC OPERATIONS
# =================================================================

import random

# ======================================================================
# IMPORT JSON FOR ARTIFACT SERIALIZATION AND HUMAN-READABLE PATH LOGGING
# ======================================================================

import json

# ===================================================================
# IMPORT OS FOR ENVIRONMENT INSPECTION AND OPTIONAL COLAB PATH CHECKS
# ===================================================================

import os

# ==========================================
# IMPORT SYS FOR PYTHON VERSION AUDIT OUTPUT
# ==========================================

import sys

# ==========================================================
# IMPORT MATH FOR SAFE LOG OPERATIONS IN ENTROPY CALCULATION
# ==========================================================

import math

# ============================================================
# IMPORT PATH FOR CROSS-PLATFORM FILE AND DIRECTORY MANAGEMENT
# ============================================================

from pathlib import Path

# ==========================================================================
# IMPORT OPTIONAL AND COLLECTION TYPE HINTS FOR READABLE FUNCTION SIGNATURES
# ==========================================================================

from typing import Dict, List, Optional, Tuple

# =============================================================
# IMPORT NUMPY FOR NUMERICAL COMPUTATION AND SQRT ANNUALIZATION
# =============================================================

import numpy as np

# ======================================================================
# IMPORT PANDAS FOR DATAFRAME TRANSFORMS AND PARQUET OR CSV INPUT OUTPUT
# ======================================================================

import pandas as pd

# =================================================================
# IMPORT MATPLOTLIB FOR REPORT-READY FIGURES SAVED AS PNG ARTIFACTS
# =================================================================

import matplotlib.pyplot as plt

# ===========================================================================
# IMPORT MATPLOTLIB PATCHES FOR DAG NODE BOX DRAWING IN THE FINAL FIGURE CELL
# ===========================================================================

from matplotlib.patches import FancyBboxPatch

# ====================================================
# IMPORT DISPLAY FOR NOTEBOOK-FRIENDLY TABLE RENDERING
# ====================================================

from IPython.display import display

# ===================================================================
# IMPORT XGBOOST REGRESSOR FOR DAG-CONSTRAINED RISK FORECAST MODELING
# ===================================================================

from xgboost import XGBRegressor

# ================================================================
# IMPORT SKLEARN ERROR METRICS FOR MAE AND ROOT MEAN SQUARED ERROR
# ================================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

# ====================================================================
# SET GLOBAL RANDOM SEED TO MATCH THE CAPSTONE TRACEABILITY CONVENTION
# ====================================================================

RANDOM_SEED = 692

# ===============================================================
# APPLY RANDOM SEED TO NUMPY FOR DETERMINISTIC NUMERICAL SAMPLING
# ===============================================================

np.random.seed(RANDOM_SEED)

# ===================================================================
# APPLY RANDOM SEED TO PYTHON RANDOM FOR DETERMINISTIC PARAMETER HANDLING
# ===================================================================

random.seed(RANDOM_SEED)

# =========================================================================
# FREEZE FORECAST HORIZON AT TWENTY TRADING DAYS TO MATCH NOTEBOOK 03
# =========================================================================

FORECAST_HORIZON_DAYS = 20

# ===========================================================================
# FREEZE WALK-FORWARD STEP AT TWENTY TRADING DAYS TO MATCH NOTEBOOK 03
# ===========================================================================

WALK_FORWARD_STEP_DAYS = 20

# ===========================================================================
# DEFINE ANNUALIZATION FACTOR USING SQUARE ROOT OF TWO HUNDRED FIFTY TWO
# ===========================================================================

ANNUALIZATION_FACTOR = float(np.sqrt(252.0))

# ========================================================================
# CONFIGURE PANDAS DISPLAY ROW LIMIT FOR AUDITABLE NOTEBOOK TABLE PREVIEWS
# ========================================================================

pd.set_option("display.max_rows", 30)

# =====================================================================
# CONFIGURE PANDAS DISPLAY COLUMN LIMIT FOR WIDE FEATURE TABLE PREVIEWS
# =====================================================================

pd.set_option("display.max_columns", 40)

# =====================================================================
# CONFIGURE CONSISTENT FLOAT DISPLAY FORMAT FOR METRIC COMPARISON CELLS
# =====================================================================

pd.set_option("display.float_format", lambda x: f"{x:0.6f}")

# =====================================================================
# CAPTURE CURRENT WORKING DIRECTORY AS THE FIRST PROJECT ROOT CANDIDATE
# =====================================================================

CWD = Path.cwd().resolve()

# =============================================================================
# ASSEMBLE FALLBACK ROOT CANDIDATES FOR LOCAL REPOSITORY AND GOOGLE COLAB PATHS
# =============================================================================

PROJECT_ROOT_CANDIDATES = []
PROJECT_ROOT_CANDIDATES.extend([CWD, *list(CWD.parents)])
PROJECT_ROOT_CANDIDATES.extend(
    [
        Path("/content/riskml-capstone"),
        Path("/content/drive/MyDrive/MScFE_692_Capstone/riskml_repo"),
        Path("/content/drive/MyDrive/riskml-capstone"),
    ]
)

# ================================================================================
# DETECT PROJECT ROOT BY FINDING THE FIRST CANDIDATE CONTAINING THE DATA DIRECTORY
# ================================================================================

PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if (path / "data").exists()), None)

# ==========================================================
# RAISE A CLEAR FILE ERROR WHEN PROJECT ROOT DETECTION FAILS
# ==========================================================

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "PROJECT ROOT DETECTION FAILED: EXPECTED A REPOSITORY ROOT CONTAINING DATA DIRECTORY"
    )

# ===============================================================================
# DEFINE CANONICAL DATA INPUT PATHS USED BY NOTEBOOK 02 AND NOTEBOOK 03 ARTIFACTS
# ===============================================================================

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_PATH = DATA_PROCESSED_DIR / "features.parquet"
TARGET_PATH = DATA_PROCESSED_DIR / "target_fwd_vol.parquet"
RETURNS_PATH = DATA_PROCESSED_DIR / "etf_returns.parquet"

# ===============================================================================
# DEFINE CANONICAL NOTEBOOK 03 ARTIFACT PATHS REQUIRED FOR NOTEBOOK 04 COMPARISON
# ===============================================================================

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"
MODELS_DIR = REPORTS_DIR / "models"
NB03_BASELINE_METRICS_PATH = TABLES_DIR / "notebook03_baseline_rmse_mae.csv"
NB03_BASELINE_PREDICTIONS_PATH = TABLES_DIR / "notebook03_baseline_predictions_long.csv"
NB03_BASELINE_TUNING_PATH = TABLES_DIR / "notebook03_xgb_tuning_results.csv"
NB03_BASELINE_IMPORTANCE_PATH = TABLES_DIR / "notebook03_xgb_feature_importance.csv"
NB03_BASELINE_MODEL_PATH = MODELS_DIR / "notebook03_xgb_baseline.json"

# =========================================================================
# DEFINE NOTEBOOK 04 OUTPUT PATHS FOR TABLES FIGURES MODELS AND PREDICTIONS
# =========================================================================

NB04_CAUSAL_METRICS_PATH = TABLES_DIR / "notebook04_causal_rmse_mae.csv"
NB04_CAUSAL_PREDICTIONS_TABLE_PATH = TABLES_DIR / "notebook04_causal_predictions_long.csv"
NB04_CAUSAL_PREDICTIONS_DATA_PATH = DATA_PROCESSED_DIR / "notebook04_causal_predictions_long.csv"
NB04_GATING_SUMMARY_PATH = TABLES_DIR / "notebook04_dag_gating_summary.csv"
NB04_EFFECTIVE_WINDOW_PATH = TABLES_DIR / "notebook04_effective_window_summary.csv"
NB04_SPLIT_SUMMARY_PATH = TABLES_DIR / "notebook04_split_summary.csv"
NB04_COMPARISON_PATH = TABLES_DIR / "causal_vs_baseline_rmse_mae.csv"
NB04_FEATURE_IMPORTANCE_PATH = TABLES_DIR / "notebook04_causal_feature_importance.csv"
NB04_ENTROPY_PATH = TABLES_DIR / "notebook04_entropy_comparison.csv"
NB04_PREFIX_GAIN_PATH = TABLES_DIR / "notebook04_prefix_gain_share.csv"
NB04_TOP_FEATURE_COMPARISON_PATH = TABLES_DIR / "notebook04_top_feature_comparison.csv"
NB04_ASSET_CLASS_PATH = TABLES_DIR / "notebook04_asset_class_comparison.csv"
NB04_ARTIFACT_MANIFEST_PATH = TABLES_DIR / "notebook04_artifact_manifest.csv"

# ===================================================================
# CREATE REPORT OUTPUT DIRECTORIES WITH IDEMPOTENT DIRECTORY CREATION
# ===================================================================

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# =======================================================================
# DEFINE ROOT MEAN SQUARED ERROR HELPER FOR CONSISTENT METRIC COMPUTATION
# =======================================================================

def compute_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # =========================================
    # RETURN ROOT MEAN SQUARED ERROR AS A FLOAT
    # =========================================

    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# =========================================================================
# DEFINE OPTIONAL CSV LOADER THAT RETURNS NONE WHEN THE FILE DOES NOT EXIST
# =========================================================================

def load_optional_csv(path: Path) -> Optional[pd.DataFrame]:
    # ======================================================
    # RETURN DATAFRAME WHEN THE CSV EXISTS OR NONE OTHERWISE
    # ======================================================

    return pd.read_csv(path) if path.exists() else None

# =========================================================================
# DEFINE CSV EXPORT HELPER WITH DIRECTORY SAFETY FOR REPORT TABLE ARTIFACTS
# =========================================================================

def save_dataframe_csv(df: pd.DataFrame, path: Path) -> None:
    # ================================================
    # ENSURE PARENT DIRECTORY EXISTS BEFORE CSV EXPORT
    # ================================================

    path.parent.mkdir(parents=True, exist_ok=True)

    # ===============================================================
    # WRITE DATAFRAME TO CSV WITHOUT INDEX FOR CLEAN REPORT INGESTION
    # ===============================================================

    df.to_csv(path, index=False)

# ============================================================================
# DEFINE PNG EXPORT HELPER WITH REPORT-READY RESOLUTION AND TIGHT BOUNDING BOX
# ============================================================================

def save_figure_png(fig: plt.Figure, path: Path, dpi: int = 200) -> None:
    # ===================================================
    # ENSURE PARENT DIRECTORY EXISTS BEFORE FIGURE EXPORT
    # ===================================================

    path.parent.mkdir(parents=True, exist_ok=True)

    # =============================================================
    # SAVE FIGURE TO PNG WITH TIGHT BOUNDING BOX FOR WORD INSERTION
    # =============================================================

    fig.savefig(path, dpi=dpi, bbox_inches="tight")

# ======================================================================
# PRINT ENVIRONMENT AND PATH AUDIT OUTPUT FOR THE FIRST NOTEBOOK 04 CELL
# ======================================================================

print("PYTHON_VERSION:", sys.version.split()[0])
print("CURRENT_WORKING_DIRECTORY:", str(CWD))
print("PROJECT_ROOT:", str(PROJECT_ROOT))
print("FEATURES_PATH_EXISTS:", FEATURES_PATH.exists())
print("TARGET_PATH_EXISTS:", TARGET_PATH.exists())
print("RETURNS_PATH_EXISTS:", RETURNS_PATH.exists())
print("NB03_BASELINE_METRICS_PATH_EXISTS:", NB03_BASELINE_METRICS_PATH.exists())
print("NB03_BASELINE_PREDICTIONS_PATH_EXISTS:", NB03_BASELINE_PREDICTIONS_PATH.exists())
print("NB03_BASELINE_IMPORTANCE_PATH_EXISTS:", NB03_BASELINE_IMPORTANCE_PATH.exists())

***

In [ ]:
# ============
# CODE_BLOCK_B
# ============

# ==================================================================
# LOAD FEATURES TARGETS AND RETURNS GENERATED BY THE PRIOR NOTEBOOKS
# ==================================================================

FEATURES_DF = pd.read_parquet(FEATURES_PATH, engine="pyarrow")
TARGET_DF = pd.read_parquet(TARGET_PATH, engine="pyarrow")
RETURNS_DF = pd.read_parquet(RETURNS_PATH, engine="pyarrow")

# ==================================================================
# LOAD OPTIONAL NOTEBOOK 03 ARTIFACTS FOR DIRECT BASELINE COMPARISON
# ==================================================================

NB03_BASELINE_METRICS_DF = load_optional_csv(NB03_BASELINE_METRICS_PATH)
NB03_BASELINE_PREDICTIONS_DF = load_optional_csv(NB03_BASELINE_PREDICTIONS_PATH)
NB03_BASELINE_TUNING_DF = load_optional_csv(NB03_BASELINE_TUNING_PATH)
NB03_BASELINE_IMPORTANCE_DF = load_optional_csv(NB03_BASELINE_IMPORTANCE_PATH)

# ========================================================================
# COERCE ALL PRIMARY INDICES TO DATETIME FOR TIME-ORDERED ALIGNMENT CHECKS
# ========================================================================

FEATURES_DF.index = pd.to_datetime(FEATURES_DF.index)
TARGET_DF.index = pd.to_datetime(TARGET_DF.index)
RETURNS_DF.index = pd.to_datetime(RETURNS_DF.index)

# ==============================================================
# SORT ALL PRIMARY DATAFRAMES TO ENFORCE MONOTONIC TIME ORDERING
# ==============================================================

FEATURES_DF = FEATURES_DF.sort_index()
TARGET_DF = TARGET_DF.sort_index()
RETURNS_DF = RETURNS_DF.sort_index()

# =============================================================================
# ASSERT INDEX ALIGNMENT ACROSS FEATURES TARGETS AND RETURNS TO PREVENT LEAKAGE
# =============================================================================

if not FEATURES_DF.index.equals(TARGET_DF.index):
    raise ValueError("INDEX MISALIGNMENT: FEATURES_DF INDEX DOES NOT MATCH TARGET_DF INDEX")

if not FEATURES_DF.index.equals(RETURNS_DF.index):
    raise ValueError("INDEX MISALIGNMENT: FEATURES_DF INDEX DOES NOT MATCH RETURNS_DF INDEX")

# =======================================================================
# IDENTIFY SENTIMENT PLACEHOLDER COLUMNS FOR EXPLICIT AUDIT AND EXCLUSION
# =======================================================================

SENTIMENT_COLS = [col for col in FEATURES_DF.columns if col.startswith("SENT__")]
SENTIMENT_NAN_FRACTION = float(FEATURES_DF[SENTIMENT_COLS].isna().mean().mean()) if len(SENTIMENT_COLS) > 0 else 0.0

# ===========================================================================
# DROP SENTIMENT PLACEHOLDER COLUMNS TO REUSE NOTEBOOK 03 MODELING CONVENTION
# ===========================================================================

FEATURES_NO_SENT_DF = FEATURES_DF.drop(columns=SENTIMENT_COLS, errors="ignore")

# ============================================================
# CHECK FOR FORBIDDEN TARGET COLUMNS INSIDE THE FEATURE MATRIX
# ============================================================

LEAKAGE_COLUMNS = [col for col in FEATURES_NO_SENT_DF.columns if col.startswith("TARGET__")]

if len(LEAKAGE_COLUMNS) > 0:
    raise ValueError(f"LEAKAGE DETECTED: TARGET-PREFIXED COLUMNS FOUND IN FEATURES: {LEAKAGE_COLUMNS[:10]}")

# =======================================================================
# CAPTURE TICKER LIST FROM RETURNS COLUMNS FOR CONSISTENT ITERATION ORDER
# =======================================================================

TICKER_LIST = list(RETURNS_DF.columns)

# =====================================================================
# SUMMARIZE FEATURE PREFIX DISTRIBUTION BEFORE EFFECTIVE WINDOW MASKING
# =====================================================================

PREFIX_COUNTS_BEFORE_MASK_DF = (
    FEATURES_NO_SENT_DF.columns.to_series()
    .map(lambda col: col.split("__")[0] if "__" in col else "UNSCOPED")
    .value_counts()
    .rename_axis("dag_prefix")
    .reset_index(name="feature_count")
    .sort_values(["dag_prefix"])
    .reset_index(drop=True)
)

# ================================================================
# PRINT INPUT SHAPES DATE RANGE AND OPTIONAL ARTIFACT AVAILABILITY
# ================================================================

print("FEATURES_DF_SHAPE:", FEATURES_DF.shape)
print("FEATURES_NO_SENT_DF_SHAPE:", FEATURES_NO_SENT_DF.shape)
print("TARGET_DF_SHAPE:", TARGET_DF.shape)
print("RETURNS_DF_SHAPE:", RETURNS_DF.shape)
print("DATE_RANGE_START:", str(FEATURES_DF.index.min().date()))
print("DATE_RANGE_END:", str(FEATURES_DF.index.max().date()))
print("SENTIMENT_COLS:", SENTIMENT_COLS)
print("SENTIMENT_NAN_FRACTION:", f"{SENTIMENT_NAN_FRACTION:0.6f}")
print("TICKER_LIST:", TICKER_LIST)
print("NB03_BASELINE_METRICS_AVAILABLE:", NB03_BASELINE_METRICS_DF is not None)
print("NB03_BASELINE_PREDICTIONS_AVAILABLE:", NB03_BASELINE_PREDICTIONS_DF is not None)
print("NB03_BASELINE_TUNING_AVAILABLE:", NB03_BASELINE_TUNING_DF is not None)
print("NB03_BASELINE_IMPORTANCE_AVAILABLE:", NB03_BASELINE_IMPORTANCE_DF is not None)

# ============================================================================
# DISPLAY SMALL TABLE PREVIEWS FOR HUMAN AUDIT AND NEXT MARKDOWN INSIGHT CELLS
# ============================================================================

display(FEATURES_NO_SENT_DF.iloc[:3, :10])
display(TARGET_DF.iloc[:3, :5])
display(RETURNS_DF.iloc[:3, :5])
display(PREFIX_COUNTS_BEFORE_MASK_DF)

***

In [ ]:
# ============
# CODE_BLOCK_C
# ============

# =========================================================================
# COMPUTE ROW-WISE COMPLETENESS MASK FOR FEATURES AFTER SENTIMENT EXCLUSION
# =========================================================================

FEATURES_COMPLETE_MASK = FEATURES_NO_SENT_DF.notna().all(axis=1)

# =============================================================
# COMPUTE ROW-WISE COMPLETENESS MASK FOR THE FULL TARGET MATRIX
# =============================================================

TARGET_COMPLETE_MASK = TARGET_DF.notna().all(axis=1)

# ====================================================================
# INTERSECT FEATURE AND TARGET COMPLETENESS MASKS TO MATCH NOTEBOOK 03
# ====================================================================

EFFECTIVE_MODELING_MASK = FEATURES_COMPLETE_MASK & TARGET_COMPLETE_MASK

# =================================================================
# APPLY THE EFFECTIVE MODELING MASK TO FEATURES TARGETS AND RETURNS
# =================================================================

EFF_FEATURES_DF = FEATURES_NO_SENT_DF.loc[EFFECTIVE_MODELING_MASK].copy()
EFF_TARGET_DF = TARGET_DF.loc[EFFECTIVE_MODELING_MASK].copy()
EFF_RETURNS_DF = RETURNS_DF.loc[EFFECTIVE_MODELING_MASK].copy()

# =========================================================================
# EXTRACT EFFECTIVE WINDOW DATES AND COUNTS FOR THE NOTEBOOK 04 AUDIT TABLE
# =========================================================================

EFFECTIVE_START_DATE = EFF_FEATURES_DF.index.min()
EFFECTIVE_END_DATE = EFF_FEATURES_DF.index.max()
EFFECTIVE_ROW_COUNT = int(EFF_FEATURES_DF.shape[0])
EFFECTIVE_FEATURE_COUNT = int(EFF_FEATURES_DF.shape[1])
EFFECTIVE_TARGET_COUNT = int(EFF_TARGET_DF.shape[1])
EFFECTIVE_FEATURES_MEAN_NON_NULL = float(EFF_FEATURES_DF.notna().mean().mean())
EFFECTIVE_TARGETS_MEAN_NON_NULL = float(EFF_TARGET_DF.notna().mean().mean())

# =====================================================================
# BUILD TICKER-TO-TARGET-COLUMN MAP FOR SAFE DAG-PREFIXED TARGET ACCESS
# =====================================================================

TARGET_COL_MAP: Dict[str, str] = {}

for ticker in TICKER_LIST:
    # ======================================
    # MATCH DAG-PREFIXED TARGET COLUMN FIRST
    # ======================================

    matched_cols = [col for col in EFF_TARGET_DF.columns if f"TARGET__{ticker}__" in col]

    # ==================================================================
    # ASSIGN THE FIRST MATCHED DAG-PREFIXED TARGET COLUMN WHEN AVAILABLE
    # ==================================================================

    if len(matched_cols) > 0:
        TARGET_COL_MAP[ticker] = matched_cols[0]

    # ==========================================================================
    # FALL BACK TO SIMPLE TICKER COLUMN NAME WHEN THE TARGET IS NOT DAG-PREFIXED
    # ==========================================================================

    elif ticker in EFF_TARGET_DF.columns:
        TARGET_COL_MAP[ticker] = ticker

    # ====================================================
    # RAISE AN ERROR WHEN NO TARGET COLUMN CAN BE RESOLVED
    # ====================================================

    else:
        raise KeyError(f"TARGET COLUMN NOT FOUND FOR TICKER={ticker}")

# ====================================================================
# SUMMARIZE FEATURE PREFIX DISTRIBUTION AFTER EFFECTIVE WINDOW MASKING
# ====================================================================

PREFIX_COUNTS_EFFECTIVE_DF = (
    EFF_FEATURES_DF.columns.to_series()
    .map(lambda col: col.split("__")[0] if "__" in col else "UNSCOPED")
    .value_counts()
    .rename_axis("dag_prefix")
    .reset_index(name="feature_count")
    .sort_values(["dag_prefix"])
    .reset_index(drop=True)
)

# =========================================================================
# BUILD EFFECTIVE WINDOW SUMMARY TABLE FOR NOTEBOOK 04 ARTIFACT PERSISTENCE
# =========================================================================

EFFECTIVE_WINDOW_SUMMARY_DF = pd.DataFrame(
    [
        {
            "effective_start_date": str(EFFECTIVE_START_DATE.date()),
            "effective_end_date": str(EFFECTIVE_END_DATE.date()),
            "effective_row_count": EFFECTIVE_ROW_COUNT,
            "effective_feature_count_ex_sentiment": EFFECTIVE_FEATURE_COUNT,
            "effective_target_count": EFFECTIVE_TARGET_COUNT,
            "effective_features_mean_non_null_fraction": EFFECTIVE_FEATURES_MEAN_NON_NULL,
            "effective_targets_mean_non_null_fraction": EFFECTIVE_TARGETS_MEAN_NON_NULL,
        }
    ]
)

# ==================================================================
# SAVE EFFECTIVE WINDOW SUMMARY FOR DOWNSTREAM NOTEBOOK TRACEABILITY
# ==================================================================

save_dataframe_csv(EFFECTIVE_WINDOW_SUMMARY_DF, NB04_EFFECTIVE_WINDOW_PATH)

# ================================================================
# PRINT EFFECTIVE WINDOW AUDIT OUTPUT AND TARGET COLUMN MAP SAMPLE
# ================================================================

print("EFFECTIVE_START_DATE:", str(EFFECTIVE_START_DATE.date()))
print("EFFECTIVE_END_DATE:", str(EFFECTIVE_END_DATE.date()))
print("EFFECTIVE_ROW_COUNT:", EFFECTIVE_ROW_COUNT)
print("EFFECTIVE_FEATURE_COUNT_EX_SENTIMENT:", EFFECTIVE_FEATURE_COUNT)
print("EFFECTIVE_TARGET_COUNT:", EFFECTIVE_TARGET_COUNT)
print("NB04_EFFECTIVE_WINDOW_PATH:", str(NB04_EFFECTIVE_WINDOW_PATH))
print("TARGET_COL_MAP:", json.dumps(TARGET_COL_MAP, indent=2))

# ==============================================================================
# DISPLAY EFFECTIVE WINDOW SUMMARY AND PREFIX COUNT TABLES FOR NOTEBOOK INSIGHTS
# ==============================================================================

display(EFFECTIVE_WINDOW_SUMMARY_DF)
display(PREFIX_COUNTS_EFFECTIVE_DF)

***

In [ ]:
# ============
# CODE_BLOCK_D
# ============

# =====================================================================
# DEFINE THE MANUAL DAG EDGE LIST AS A DIRECTIONAL CONSTRAINT REFERENCE
# =====================================================================

CAUSAL_EDGES = {
    "SENTIMENT": ["MOMENTUM"],
    "MOMENTUM": ["RETURNS"],
    "VALUE": ["RETURNS"],
    "VOLATILITY": ["RISK"],
    "RISK": ["ALLOCATION"],
}

# ================================================================
# DEFINE PREFIX MAP USED TO TRANSLATE FEATURE NAMES INTO DAG NODES
# ================================================================

NODE_PREFIX_MAP = {
    "SENTIMENT": "SENT__",
    "MOMENTUM": "MOM__",
    "VALUE": "VAL__",
    "VOLATILITY": "VOL__",
    "MACRO": "MACRO__",
    "ML": "ML__",
    "REGIME": "REGIME__",
    "TARGET": "TARGET__",
}

# ====================================================================
# FREEZE ALLOWED AND FORBIDDEN PREFIX SETS FOR THE RISK FORECAST STAGE
# ====================================================================

ALLOWED_PREFIXES_RISK = ["VOL__", "MACRO__", "REGIME__"]
FORBIDDEN_PREFIXES_RISK = ["MOM__", "VAL__", "ML__", "SENT__"]

# ========================================================================
# DEFINE HELPER TO EXTRACT THE DAG PREFIX TOKEN FROM A FEATURE COLUMN NAME
# ========================================================================

def extract_feature_prefix(column_name: str) -> str:
    # ==============================================================================
    # RETURN THE FIRST DOUBLE-UNDERSCORE TOKEN OR UNSCOPED WHEN THE TOKEN IS MISSING
    # ==============================================================================

    return column_name.split("__")[0] if "__" in column_name else "UNSCOPED"

# ========================================================
# DEFINE HELPER TO GATE FEATURES BY AN ALLOWED PREFIX LIST
# ========================================================

def gate_features_by_prefix(columns: List[str], allowed_prefixes: List[str]) -> Tuple[List[str], List[str]]:
    # ========================================================
    # COLLECT ALLOWED FEATURE COLUMNS IN ORIGINAL COLUMN ORDER
    # ========================================================

    allowed_cols = [col for col in columns if any(col.startswith(prefix) for prefix in allowed_prefixes)]

    # =========================================================
    # COLLECT EXCLUDED FEATURE COLUMNS IN ORIGINAL COLUMN ORDER
    # =========================================================

    excluded_cols = [col for col in columns if col not in allowed_cols]

    # =============================================
    # RETURN BOTH ALLOWED AND EXCLUDED COLUMN LISTS
    # =============================================

    return allowed_cols, excluded_cols

# =====================================================================
# APPLY PREFIX GATING TO THE EFFECTIVE FEATURE MATRIX FOR THE RISK NODE
# =====================================================================

CAUSAL_FEATURE_COLS, EXCLUDED_FEATURE_COLS = gate_features_by_prefix(list(EFF_FEATURES_DF.columns), ALLOWED_PREFIXES_RISK)

# ====================================================================
# VALIDATE THAT NO FORBIDDEN PREFIX SURVIVED THE RISK NODE GATING STEP
# ====================================================================

LEAKED_FORBIDDEN_COLS = [
    col
    for col in CAUSAL_FEATURE_COLS
    if any(col.startswith(prefix) for prefix in FORBIDDEN_PREFIXES_RISK)
]

if len(LEAKED_FORBIDDEN_COLS) > 0:
    raise ValueError(f"DAG GATING FAILURE: FORBIDDEN FEATURES LEAKED INTO RISK STAGE: {LEAKED_FORBIDDEN_COLS[:10]}")

# ======================================================================
# BUILD GATING SUMMARY TABLE WITH PREFIX COUNTS AND ALLOW EXCLUDE STATUS
# ======================================================================

GATING_SUMMARY_DF = (
    pd.DataFrame({"feature": list(EFF_FEATURES_DF.columns)})
    .assign(
        dag_prefix=lambda df: df["feature"].map(extract_feature_prefix),
        allowed_for_risk=lambda df: df["feature"].isin(CAUSAL_FEATURE_COLS),
    )
)

GATING_PREFIX_SUMMARY_DF = (
    GATING_SUMMARY_DF.groupby(["dag_prefix", "allowed_for_risk"], dropna=False)["feature"]
    .count()
    .reset_index(name="feature_count")
    .sort_values(["allowed_for_risk", "dag_prefix"], ascending=[False, True])
    .reset_index(drop=True)
)

# ==================================================================
# SAVE THE FEATURE-LEVEL GATING SUMMARY FOR NOTEBOOK 04 TRACEABILITY
# ==================================================================

save_dataframe_csv(GATING_SUMMARY_DF, NB04_GATING_SUMMARY_PATH)

# ====================================================================
# CREATE THE CONSTRAINED FEATURE MATRIX USED BY ALL CAUSAL MODEL CELLS
# ====================================================================

CAUSAL_X_DF = EFF_FEATURES_DF.loc[:, CAUSAL_FEATURE_COLS].copy()

# ===================================================================
# PRINT GATING COUNTS AND EXPECTED PREFIX COUNTS FOR QUICK VALIDATION
# ===================================================================

print("CAUSAL_FEATURE_COUNT:", len(CAUSAL_FEATURE_COLS))
print("EXCLUDED_FEATURE_COUNT:", len(EXCLUDED_FEATURE_COLS))
print("ALLOWED_PREFIXES_RISK:", ALLOWED_PREFIXES_RISK)
print("FORBIDDEN_PREFIXES_RISK:", FORBIDDEN_PREFIXES_RISK)
print("LEAKED_FORBIDDEN_COLS:", LEAKED_FORBIDDEN_COLS)
print("NB04_GATING_SUMMARY_PATH:", str(NB04_GATING_SUMMARY_PATH))

# ===============================================================
# DISPLAY PREFIX SUMMARY AND EXAMPLE ALLOWED OR EXCLUDED FEATURES
# ===============================================================

display(GATING_PREFIX_SUMMARY_DF)
display(pd.DataFrame({"allowed_feature_example": CAUSAL_FEATURE_COLS[:20]}))
display(pd.DataFrame({"excluded_feature_example": EXCLUDED_FEATURE_COLS[:20]}))

***

In [ ]:
# ============
# CODE_BLOCK_E
# ============

# ===================================================================
# COMPUTE EFFECTIVE SAMPLE LENGTH FOR THE TRAIN VALIDATION TEST SPLIT
# ===================================================================

N_EFFECTIVE = int(len(CAUSAL_X_DF))

# ===============================================================
# RECREATE THE NOTEBOOK 03 SIXTY TWENTY TWENTY TIME-ORDERED SPLIT
# ===============================================================

TRAIN_SIZE = int(np.floor(0.60 * N_EFFECTIVE))
VAL_SIZE = int(np.floor(0.20 * N_EFFECTIVE))
TEST_SIZE = int(N_EFFECTIVE - TRAIN_SIZE - VAL_SIZE)

# ==================================================================
# DEFINE POSITIONAL BOUNDARIES FOR TRAIN VALIDATION AND TEST WINDOWS
# ==================================================================

TRAIN_START_IDX = 0
TRAIN_END_IDX = TRAIN_START_IDX + TRAIN_SIZE - 1
VAL_START_IDX = TRAIN_END_IDX + 1
VAL_END_IDX = VAL_START_IDX + VAL_SIZE - 1
TEST_START_IDX = VAL_END_IDX + 1
TEST_END_IDX = N_EFFECTIVE - 1

# ========================================================
# EXTRACT DATE BOUNDARIES FOR AUDITABLE SPLIT PRINT OUTPUT
# ========================================================

TRAIN_START_DATE = CAUSAL_X_DF.index[TRAIN_START_IDX]
TRAIN_END_DATE = CAUSAL_X_DF.index[TRAIN_END_IDX]
VAL_START_DATE = CAUSAL_X_DF.index[VAL_START_IDX]
VAL_END_DATE = CAUSAL_X_DF.index[VAL_END_IDX]
TEST_START_DATE = CAUSAL_X_DF.index[TEST_START_IDX]
TEST_END_DATE = CAUSAL_X_DF.index[TEST_END_IDX]

# =================================================================
# BUILD EXPANDING-WINDOW BLOCK START INDICES USING TWENTY-DAY STEPS
# =================================================================

VAL_BLOCK_START_IDXS = list(range(VAL_START_IDX, TEST_START_IDX, WALK_FORWARD_STEP_DAYS))
TEST_BLOCK_START_IDXS = list(range(TEST_START_IDX, N_EFFECTIVE, WALK_FORWARD_STEP_DAYS))

# ======================================================================
# DEFINE LABEL MATURITY RULE THAT PROTECTS THE TWENTY-DAY FORWARD TARGET
# ======================================================================

def training_end_index_for_block(block_start_idx: int, horizon_days: int) -> int:
    # ===========================================================================
    # RETURN THE LAST INDEX WITH A MATURE LABEL AVAILABLE AT THE BLOCK START DATE
    # ===========================================================================

    return int(block_start_idx - horizon_days)

# =======================================================================
# BUILD SPLIT SUMMARY TABLE FOR NOTEBOOK 04 TRACEABILITY AND REPORT REUSE
# =======================================================================

SPLIT_SUMMARY_DF = pd.DataFrame(
    [
        {"split": "TRAIN", "start_date": str(TRAIN_START_DATE.date()), "end_date": str(TRAIN_END_DATE.date()), "row_count": TRAIN_SIZE},
        {"split": "VALIDATION", "start_date": str(VAL_START_DATE.date()), "end_date": str(VAL_END_DATE.date()), "row_count": VAL_SIZE},
        {"split": "TEST", "start_date": str(TEST_START_DATE.date()), "end_date": str(TEST_END_DATE.date()), "row_count": TEST_SIZE},
    ]
)

# ==========================================================================
# SAVE SPLIT SUMMARY TABLE FOR DOWNSTREAM NOTEBOOK 05 AND REPORT REFERENCING
# ==========================================================================

save_dataframe_csv(SPLIT_SUMMARY_DF, NB04_SPLIT_SUMMARY_PATH)

# =========================================================================
# PRINT SPLIT COUNTS BLOCK COUNTS AND SHAPES FOR NOTEBOOK VALIDATION OUTPUT
# =========================================================================

print("N_EFFECTIVE:", N_EFFECTIVE)
print("CAUSAL_X_DF_SHAPE:", CAUSAL_X_DF.shape)
print("TRAIN_SIZE:", TRAIN_SIZE, "|", "TRAIN_START_DATE:", str(TRAIN_START_DATE.date()), "|", "TRAIN_END_DATE:", str(TRAIN_END_DATE.date()))
print("VAL_SIZE:", VAL_SIZE, "|", "VAL_START_DATE:", str(VAL_START_DATE.date()), "|", "VAL_END_DATE:", str(VAL_END_DATE.date()))
print("TEST_SIZE:", TEST_SIZE, "|", "TEST_START_DATE:", str(TEST_START_DATE.date()), "|", "TEST_END_DATE:", str(TEST_END_DATE.date()))
print("VAL_BLOCK_COUNT:", len(VAL_BLOCK_START_IDXS))
print("TEST_BLOCK_COUNT:", len(TEST_BLOCK_START_IDXS))
print("NB04_SPLIT_SUMMARY_PATH:", str(NB04_SPLIT_SUMMARY_PATH))

# =========================================================================
# DISPLAY SPLIT SUMMARY AND THE FIRST FEW VALIDATION TEST BLOCK START DATES
# =========================================================================

display(SPLIT_SUMMARY_DF)
display(pd.DataFrame({"val_block_start_dates": [CAUSAL_X_DF.index[idx] for idx in VAL_BLOCK_START_IDXS[:5]]}))
display(pd.DataFrame({"test_block_start_dates": [CAUSAL_X_DF.index[idx] for idx in TEST_BLOCK_START_IDXS[:5]]}))

***

In [ ]:
# ============
# CODE_BLOCK_F
# ============

# ======================================================================
# FREEZE THE NOTEBOOK 03 XGBOOST HYPERPARAMETERS FOR FAIR DAG COMPARISON
# ======================================================================

FROZEN_XGB_PARAMS = {
    "n_estimators": 400,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_lambda": 1.0,
    "tree_method": "hist",
}

# ============================================================================
# DEFINE XGBOOST MODEL FACTORY WITH FROZEN PARAMETERS AND REPRODUCIBLE SEEDING
# ============================================================================

def build_xgb_regressor(params: Dict[str, float]) -> XGBRegressor:
    # ==========================================================================
    # RETURN A SQUARED-ERROR XGBOOST REGRESSOR FOR CONTINUOUS VOLATILITY TARGETS
    # ==========================================================================

    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=int(params["n_estimators"]),
        max_depth=int(params["max_depth"]),
        learning_rate=float(params["learning_rate"]),
        subsample=float(params["subsample"]),
        colsample_bytree=float(params["colsample_bytree"]),
        reg_lambda=float(params.get("reg_lambda", 1.0)),
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method=str(params.get("tree_method", "hist")),
    )

# =================================================================================
# DEFINE WALK-FORWARD PREDICTION FUNCTION USING THE NOTEBOOK 03 LABEL-SAFE PROTOCOL
# =================================================================================

def walk_forward_predict_xgb(
    x_df: pd.DataFrame,
    y_series: pd.Series,
    block_start_idxs: List[int],
    horizon_days: int,
    step_days: int,
    model_params: Dict[str, float],
) -> pd.DataFrame:
    # ========================================================
    # INITIALIZE ROW CONTAINER FOR LONG-FORM PREDICTION OUTPUT
    # ========================================================

    rows: List[Dict[str, float]] = []

    # =====================================================================
    # LOOP OVER TEST BLOCK START INDICES TO EMULATE EXPANDING-WINDOW REFITS
    # =====================================================================

    for block_start_idx in block_start_idxs:
        # ===========================================================
        # COMPUTE THE LAST TRAINING INDEX WITH A MATURE FORWARD LABEL
        # ===========================================================

        train_end_idx = training_end_index_for_block(block_start_idx, horizon_days)

        # ======================================================
        # SKIP THE BLOCK WHEN TOO LITTLE TRAINING HISTORY EXISTS
        # ======================================================

        if train_end_idx < 100:
            continue

        # ==================================================
        # SLICE TRAINING DATA UP TO THE LABEL-SAFE END INDEX
        # ==================================================

        x_train = x_df.iloc[: train_end_idx + 1]
        y_train = y_series.iloc[: train_end_idx + 1]

        # ============================================================
        # SLICE THE NEXT STEP-SIZED BLOCK FOR OUT-OF-SAMPLE PREDICTION
        # ============================================================

        block_end_idx_exclusive = min(block_start_idx + step_days, len(x_df))
        x_block = x_df.iloc[block_start_idx:block_end_idx_exclusive]
        y_block = y_series.iloc[block_start_idx:block_end_idx_exclusive]

        # =========================================
        # FIT A FRESH MODEL ON THE EXPANDING WINDOW
        # =========================================

        model = build_xgb_regressor(model_params)
        model.fit(x_train, y_train)

        # ========================================================
        # GENERATE PREDICTIONS FOR THE CURRENT OUT-OF-SAMPLE BLOCK
        # ========================================================

        y_pred = model.predict(x_block)

        # ===============================================================
        # APPEND ONE LONG-FORM ROW PER DATE FOR CLEAN EXPORT AND PLOTTING
        # ===============================================================

        for row_idx, dt in enumerate(x_block.index):
            rows.append(
                {
                    "date": str(pd.to_datetime(dt).date()),
                    "y_true": float(y_block.iloc[row_idx]),
                    "y_pred": float(y_pred[row_idx]),
                }
            )

    # =========================================
    # RETURN THE LONG-FORM PREDICTION DATAFRAME
    # =========================================

    return pd.DataFrame(rows)

# =====================================================================
# INITIALIZE METRIC AND PREDICTION CONTAINERS FOR THE CAUSAL MODEL LOOP
# =====================================================================

CAUSAL_METRIC_ROWS: List[Dict[str, float]] = []
CAUSAL_PREDICTION_ROWS: List[Dict[str, float]] = []

# ================================================================================
# LOOP OVER ALL TICKERS TO FIT THE DAG-CONSTRAINED MODEL AND SCORE THE TEST WINDOW
# ================================================================================

for ticker in TICKER_LIST:
    # =============================================================
    # SELECT THE GATED CAUSAL FEATURE MATRIX FOR THE CURRENT TICKER
    # =============================================================

    x_df = CAUSAL_X_DF

    # =================================================================
    # SELECT THE TARGET SERIES THROUGH THE TARGET COLUMN GOVERNANCE MAP
    # =================================================================

    y_series = EFF_TARGET_DF[TARGET_COL_MAP[ticker]]

    # ===========================================================
    # RUN LABEL-SAFE WALK-FORWARD PREDICTION OVER THE TEST BLOCKS
    # ===========================================================

    preds_df = walk_forward_predict_xgb(
        x_df=x_df,
        y_series=y_series,
        block_start_idxs=TEST_BLOCK_START_IDXS,
        horizon_days=FORECAST_HORIZON_DAYS,
        step_days=WALK_FORWARD_STEP_DAYS,
        model_params=FROZEN_XGB_PARAMS,
    )

    # ======================================================================
    # RAISE AN ERROR WHEN NO PREDICTIONS ARE PRODUCED FOR THE CURRENT TICKER
    # ======================================================================

    if preds_df.empty:
        raise ValueError(f"NO CAUSAL PREDICTIONS PRODUCED FOR TICKER={ticker}")

    # ===========================================
    # COMPUTE RMSE AND MAE FOR THE CURRENT TICKER
    # ===========================================

    rmse_val = compute_rmse(preds_df["y_true"].values, preds_df["y_pred"].values)
    mae_val = float(mean_absolute_error(preds_df["y_true"].values, preds_df["y_pred"].values))

    # ==========================================================
    # APPEND PER-TICKER METRICS FOR THE NOTEBOOK 04 METRIC TABLE
    # ==========================================================

    CAUSAL_METRIC_ROWS.append(
        {
            "model": "CAUSAL_XGBOOST",
            "ticker": ticker,
            "rmse": rmse_val,
            "mae": mae_val,
            "n_obs": int(len(preds_df)),
            "feature_count": int(x_df.shape[1]),
        }
    )

    # ======================================================
    # APPEND LONG-FORM PREDICTIONS FOR EXPORT AND PLOT REUSE
    # ======================================================

    for row in preds_df.to_dict(orient="records"):
        CAUSAL_PREDICTION_ROWS.append(
            {
                "date": row["date"],
                "ticker": ticker,
                "model": "CAUSAL_XGBOOST",
                "y_true": row["y_true"],
                "y_pred": row["y_pred"],
            }
        )

# ===========================================================================
# BUILD THE CAUSAL METRICS DATAFRAME AND APPEND AN EQUAL-WEIGHT AGGREGATE ROW
# ===========================================================================

CAUSAL_METRICS_DF = pd.DataFrame(CAUSAL_METRIC_ROWS)

CAUSAL_AGG_ROW = {
    "model": "CAUSAL_XGBOOST",
    "ticker": "AGGREGATE_EQUAL_WEIGHT",
    "rmse": float(CAUSAL_METRICS_DF["rmse"].mean()),
    "mae": float(CAUSAL_METRICS_DF["mae"].mean()),
    "n_obs": int(CAUSAL_METRICS_DF["n_obs"].min()),
    "feature_count": int(CAUSAL_METRICS_DF["feature_count"].max()),
}

CAUSAL_METRICS_DF = pd.concat([CAUSAL_METRICS_DF, pd.DataFrame([CAUSAL_AGG_ROW])], ignore_index=True)

# =====================================================================
# BUILD THE LONG-FORM CAUSAL PREDICTION DATAFRAME AND SAVE BOTH OUTPUTS
# =====================================================================

CAUSAL_PREDICTIONS_LONG_DF = pd.DataFrame(CAUSAL_PREDICTION_ROWS)
save_dataframe_csv(CAUSAL_METRICS_DF, NB04_CAUSAL_METRICS_PATH)
save_dataframe_csv(CAUSAL_PREDICTIONS_LONG_DF, NB04_CAUSAL_PREDICTIONS_TABLE_PATH)
save_dataframe_csv(CAUSAL_PREDICTIONS_LONG_DF, NB04_CAUSAL_PREDICTIONS_DATA_PATH)

# ============================================================================
# PRINT THE FROZEN PARAMETER SET OUTPUT PATHS AND THE CAUSAL AGGREGATE METRICS
# ============================================================================

print("FROZEN_XGB_PARAMS:", json.dumps(FROZEN_XGB_PARAMS, indent=2))
print("CAUSAL_FEATURE_COUNT:", CAUSAL_X_DF.shape[1])
print("NB04_CAUSAL_METRICS_PATH:", str(NB04_CAUSAL_METRICS_PATH))
print("NB04_CAUSAL_PREDICTIONS_TABLE_PATH:", str(NB04_CAUSAL_PREDICTIONS_TABLE_PATH))
print("NB04_CAUSAL_PREDICTIONS_DATA_PATH:", str(NB04_CAUSAL_PREDICTIONS_DATA_PATH))

# =================================================================
# DISPLAY THE CAUSAL METRIC TABLE AND A LONG-FORM PREDICTION SAMPLE
# =================================================================

display(CAUSAL_METRICS_DF.sort_values(["ticker"]).reset_index(drop=True))
display(CAUSAL_PREDICTIONS_LONG_DF.head(20))

***

In [ ]:
# ============
# CODE_BLOCK_G
# ============

# ================================================================================
# DEFINE HELPER TO RECOMPUTE METRICS FROM A LONG-FORM PREDICTION TABLE WHEN NEEDED
# ================================================================================

def metrics_from_long_predictions(long_df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    # ====================================================================
    # FILTER TO ONE MODEL AND ASSERT THAT THE LONG-FORM TABLE IS NOT EMPTY
    # ====================================================================

    model_df = long_df[long_df["model"] == model_name].copy()

    if model_df.empty:
        raise ValueError(f"LONG-FORM PREDICTION TABLE CONTAINS NO ROWS FOR MODEL={model_name}")

    # ==========================================================
    # GROUP BY TICKER AND COMPUTE RMSE MAE AND OBSERVATION COUNT
    # ==========================================================

    rows = []
    for ticker, ticker_df in model_df.groupby("ticker", sort=True):
        rows.append(
            {
                "model": model_name,
                "ticker": ticker,
                "rmse": compute_rmse(ticker_df["y_true"].values, ticker_df["y_pred"].values),
                "mae": float(mean_absolute_error(ticker_df["y_true"].values, ticker_df["y_pred"].values)),
                "n_obs": int(len(ticker_df)),
            }
        )

    # ====================================
    # APPEND AN EQUAL-WEIGHT AGGREGATE ROW
    # ====================================

    out_df = pd.DataFrame(rows)
    agg_row = {
        "model": model_name,
        "ticker": "AGGREGATE_EQUAL_WEIGHT",
        "rmse": float(out_df["rmse"].mean()),
        "mae": float(out_df["mae"].mean()),
        "n_obs": int(out_df["n_obs"].min()),
    }

    # =======================================================
    # RETURN THE METRIC TABLE WITH THE AGGREGATE ROW APPENDED
    # =======================================================

    return pd.concat([out_df, pd.DataFrame([agg_row])], ignore_index=True)

# ==============================================================================
# LOAD NOTEBOOK 03 METRICS WHEN AVAILABLE OR RECOMPUTE FROM BASELINE PREDICTIONS
# ==============================================================================

if NB03_BASELINE_METRICS_DF is not None:
    BASELINE_METRICS_DF = NB03_BASELINE_METRICS_DF.copy()
elif NB03_BASELINE_PREDICTIONS_DF is not None:
    EWMA_FROM_LONG_DF = metrics_from_long_predictions(NB03_BASELINE_PREDICTIONS_DF, "EWMA")
    XGBOOST_FROM_LONG_DF = metrics_from_long_predictions(NB03_BASELINE_PREDICTIONS_DF, "XGBOOST")
    BASELINE_METRICS_DF = pd.concat([EWMA_FROM_LONG_DF, XGBOOST_FROM_LONG_DF], ignore_index=True)
else:
    raise FileNotFoundError("NOTEBOOK 03 BASELINE METRICS OR PREDICTIONS ARTIFACT REQUIRED BEFORE NOTEBOOK 04 COMPARISON")

# =========================================================================
# SUBSET THE BASELINE TABLE TO THE TWO REFERENCE MODELS USED IN NOTEBOOK 04
# =========================================================================

BASELINE_METRICS_DF = BASELINE_METRICS_DF[BASELINE_METRICS_DF["model"].isin(["EWMA", "XGBOOST"])].copy()

# =========================================================
# BUILD XGBOOST AND EWMA REFERENCE TABLES FOR CLEAN MERGING
# =========================================================

BASELINE_XGB_METRICS_DF = BASELINE_METRICS_DF[BASELINE_METRICS_DF["model"] == "XGBOOST"].copy()
BASELINE_EWMA_METRICS_DF = BASELINE_METRICS_DF[BASELINE_METRICS_DF["model"] == "EWMA"].copy()

# ============================================================================
# MERGE CAUSAL METRICS AGAINST THE XGBOOST BASELINE FOR THE PRIMARY H ONE TEST
# ============================================================================

CAUSAL_VS_XGB_DF = (
    CAUSAL_METRICS_DF.rename(columns={"rmse": "causal_rmse", "mae": "causal_mae", "n_obs": "causal_n_obs"})
    .merge(
        BASELINE_XGB_METRICS_DF.rename(columns={"rmse": "xgboost_rmse", "mae": "xgboost_mae", "n_obs": "xgboost_n_obs"}).drop(columns=["model"]),
        on="ticker",
        how="left",
    )
    .assign(
        causal_minus_xgboost_rmse=lambda df: df["causal_rmse"] - df["xgboost_rmse"],
        causal_minus_xgboost_mae=lambda df: df["causal_mae"] - df["xgboost_mae"],
        rmse_improves_vs_xgboost=lambda df: df["causal_minus_xgboost_rmse"] < 0,
        mae_improves_vs_xgboost=lambda df: df["causal_minus_xgboost_mae"] < 0,
    )
)

# =========================================================
# ADD EWMA REFERENCE COLUMNS FOR SECONDARY BASELINE CONTEXT
# =========================================================

CAUSAL_VS_BASELINE_DF = (
    CAUSAL_VS_XGB_DF.merge(
        BASELINE_EWMA_METRICS_DF.rename(columns={"rmse": "ewma_rmse", "mae": "ewma_mae", "n_obs": "ewma_n_obs"}).drop(columns=["model"]),
        on="ticker",
        how="left",
    )
    .assign(
        causal_minus_ewma_rmse=lambda df: df["causal_rmse"] - df["ewma_rmse"],
        causal_minus_ewma_mae=lambda df: df["causal_mae"] - df["ewma_mae"],
        rmse_improves_vs_ewma=lambda df: df["causal_minus_ewma_rmse"] < 0,
        mae_improves_vs_ewma=lambda df: df["causal_minus_ewma_mae"] < 0,
    )
)

# ====================================================================
# SAVE THE NOTEBOOK 04 PRIMARY COMPARISON TABLE FOR REPORT INTEGRATION
# ====================================================================

save_dataframe_csv(CAUSAL_VS_BASELINE_DF, NB04_COMPARISON_PATH)

# ====================================================================
# COMPUTE SIMPLE WIN COUNTS AGAINST BOTH BASELINES FOR NOTEBOOK OUTPUT
# ====================================================================

NON_AGG_COMPARISON_DF = CAUSAL_VS_BASELINE_DF[CAUSAL_VS_BASELINE_DF["ticker"] != "AGGREGATE_EQUAL_WEIGHT"].copy()
RMSE_WINS_VS_XGB = int(NON_AGG_COMPARISON_DF["rmse_improves_vs_xgboost"].sum())
RMSE_WINS_VS_EWMA = int(NON_AGG_COMPARISON_DF["rmse_improves_vs_ewma"].sum())

# =======================================================================
# PRINT COMPARISON PATH AND WIN COUNTS FOR NOTEBOOK 04 INTERPRETIVE CELLS
# =======================================================================

print("NB04_COMPARISON_PATH:", str(NB04_COMPARISON_PATH))
print("RMSE_WINS_VS_XGBOOST:", RMSE_WINS_VS_XGB)
print("RMSE_WINS_VS_EWMA:", RMSE_WINS_VS_EWMA)

# ===============================================================
# DISPLAY THE FULL CAUSAL VERSUS BASELINE METRIC COMPARISON TABLE
# ===============================================================

display(CAUSAL_VS_BASELINE_DF.sort_values(["ticker"]).reset_index(drop=True))

***

In [ ]:
# ============
# CODE_BLOCK_H
# ============

# ==================================================================
# SELECT REPRESENTATIVE TICKERS FOR FINAL INTERPRETABILITY ARTIFACTS
# ==================================================================

REP_TICKERS_IMPORTANCE = [ticker for ticker in ["SPY", "TLT", "GLD"] if ticker in TICKER_LIST]

if len(REP_TICKERS_IMPORTANCE) == 0:
    REP_TICKERS_IMPORTANCE = TICKER_LIST[:3]

# =========================================================================
# DEFINE HELPER TO CONVERT XGBOOST GAIN DICTIONARIES INTO SORTED DATAFRAMES
# =========================================================================

def gain_dict_to_dataframe(gain_dict: Dict[str, float], ticker: str, model_variant: str) -> pd.DataFrame:
    # ===========================================================================
    # BUILD A LONG-FORM IMPORTANCE TABLE WITH PREFIX TAGS AND WITHIN-TICKER RANKS
    # ===========================================================================

    fi_df = pd.DataFrame([{"feature": feature, "gain": float(gain)} for feature, gain in gain_dict.items()])

    if fi_df.empty:
        return pd.DataFrame(columns=["ticker", "model_variant", "feature", "dag_prefix", "gain", "gain_share", "rank_within_ticker"])

    fi_df = fi_df.sort_values("gain", ascending=False).reset_index(drop=True)
    fi_df["ticker"] = ticker
    fi_df["model_variant"] = model_variant
    fi_df["dag_prefix"] = fi_df["feature"].map(extract_feature_prefix)
    fi_df["gain_share"] = fi_df["gain"] / fi_df["gain"].sum()
    fi_df["rank_within_ticker"] = np.arange(1, len(fi_df) + 1)

    # =====================================
    # RETURN THE LONG-FORM IMPORTANCE TABLE
    # =====================================

    return fi_df[["ticker", "model_variant", "feature", "dag_prefix", "gain", "gain_share", "rank_within_ticker"]]

# ======================================================================================
# FIT FINAL CONSTRAINED MODELS ON THE FULL EFFECTIVE WINDOW FOR INTERPRETABILITY EXPORTS
# ======================================================================================

FINAL_CAUSAL_MODELS: Dict[str, XGBRegressor] = {}
CAUSAL_IMPORTANCE_DFS: List[pd.DataFrame] = []

for ticker in REP_TICKERS_IMPORTANCE:
    # =================================================================================
    # SELECT THE FULL GATED FEATURE MATRIX AND THE TARGET SERIES FOR THE CURRENT TICKER
    # =================================================================================

    x_full = CAUSAL_X_DF.copy()
    y_full = EFF_TARGET_DF[TARGET_COL_MAP[ticker]].copy()

    # ===============================
    # FIT THE FINAL CONSTRAINED MODEL
    # ===============================

    final_model = build_xgb_regressor(FROZEN_XGB_PARAMS)
    final_model.fit(x_full, y_full)
    FINAL_CAUSAL_MODELS[ticker] = final_model

    # =========================================================
    # SAVE THE FINAL CONSTRAINED MODEL JSON FOR REPRODUCIBILITY
    # =========================================================

    model_path = MODELS_DIR / f"notebook04_xgb_causal_{ticker}.json"
    final_model.save_model(str(model_path))

    # ============================================================
    # EXTRACT GAIN-BASED FEATURE IMPORTANCE FOR THE CURRENT TICKER
    # ============================================================

    gain_dict = final_model.get_booster().get_score(importance_type="gain")
    causal_fi_df = gain_dict_to_dataframe(gain_dict, ticker=ticker, model_variant="CAUSAL_CONSTRAINED")
    causal_fi_df["model_path"] = str(model_path)
    CAUSAL_IMPORTANCE_DFS.append(causal_fi_df)

# ========================================================================
# CONCATENATE THE CONSTRAINED IMPORTANCE TABLES AND VALIDATE PREFIX PURITY
# ========================================================================

CAUSAL_FEATURE_IMPORTANCE_DF = pd.concat(CAUSAL_IMPORTANCE_DFS, ignore_index=True)

FORBIDDEN_IMPORTANCE_ROWS_DF = CAUSAL_FEATURE_IMPORTANCE_DF[
    CAUSAL_FEATURE_IMPORTANCE_DF["dag_prefix"].isin([prefix.replace("__", "") for prefix in FORBIDDEN_PREFIXES_RISK])
].copy()

if not FORBIDDEN_IMPORTANCE_ROWS_DF.empty:
    raise ValueError("FORBIDDEN DAG PREFIX APPEARED IN CONSTRAINED FEATURE IMPORTANCE OUTPUT")

# =====================================================================
# SAVE THE CONSTRAINED FEATURE IMPORTANCE TABLE FOR THE REPORT APPENDIX
# =====================================================================

save_dataframe_csv(CAUSAL_FEATURE_IMPORTANCE_DF, NB04_FEATURE_IMPORTANCE_PATH)

# ===========================================================
# PRINT REPRESENTATIVE TICKERS AND MODEL EXPORT PATH EXAMPLES
# ===========================================================

print("REP_TICKERS_IMPORTANCE:", REP_TICKERS_IMPORTANCE)
print("NB04_FEATURE_IMPORTANCE_PATH:", str(NB04_FEATURE_IMPORTANCE_PATH))
print("FINAL_CAUSAL_MODEL_EXPORTS:", [str(MODELS_DIR / f"notebook04_xgb_causal_{ticker}.json") for ticker in REP_TICKERS_IMPORTANCE])

# ===================================================================
# DISPLAY THE TOP CONSTRAINED FEATURES FOR EACH REPRESENTATIVE TICKER
# ===================================================================

display(CAUSAL_FEATURE_IMPORTANCE_DF.groupby("ticker", group_keys=False).head(10).reset_index(drop=True))

***

In [ ]:
# ============
# CODE_BLOCK_I
# ============

# ==============================================================================
# DEFINE HELPER TO COMPUTE ENTROPY AND CONCENTRATION STATISTICS FROM GAIN SHARES
# ==============================================================================

def compute_entropy_stats(fi_df: pd.DataFrame) -> Dict[str, float]:
    # ========================================================
    # RETURN NA-LIKE VALUES WHEN THE IMPORTANCE TABLE IS EMPTY
    # ========================================================

    if fi_df.empty:
        return {
            "entropy": float("nan"),
            "normalized_entropy": float("nan"),
            "non_zero_feature_count": 0,
            "top_1_gain_share": float("nan"),
            "top_5_gain_share": float("nan"),
        }

    # ===============================================
    # NORMALIZE GAIN VALUES INTO A PROBABILITY VECTOR
    # ===============================================

    p = fi_df["gain_share"].astype(float).values
    p = p[p > 0]

    # ================================================
    # COMPUTE SHANNON ENTROPY USING NATURAL LOGARITHMS
    # ================================================

    entropy = float(-(p * np.log(p)).sum())
    normalized_entropy = float(entropy / np.log(len(p))) if len(p) > 1 else 0.0

    # ===========================================================
    # RETURN ENTROPY PLUS SIMPLE CONCENTRATION SUMMARY STATISTICS
    # ===========================================================

    return {
        "entropy": entropy,
        "normalized_entropy": normalized_entropy,
        "non_zero_feature_count": int(len(p)),
        "top_1_gain_share": float(np.sort(p)[::-1][:1].sum()),
        "top_5_gain_share": float(np.sort(p)[::-1][:5].sum()),
    }

# ==================================================================
# FIT UNCONSTRAINED FINAL MODELS FOR THE SAME REPRESENTATIVE TICKERS
# ==================================================================

UNCONSTRAINED_IMPORTANCE_DFS: List[pd.DataFrame] = []

for ticker in REP_TICKERS_IMPORTANCE:
    # ==============================================================
    # REUSE NOTEBOOK 03 SAVED SPY IMPORTANCE ARTIFACT WHEN AVAILABLE
    # ==============================================================

    if ticker == "SPY" and NB03_BASELINE_IMPORTANCE_DF is not None:
        baseline_spy_fi_df = NB03_BASELINE_IMPORTANCE_DF.copy()
        baseline_spy_fi_df["ticker"] = "SPY"
        baseline_spy_fi_df["model_variant"] = "BASELINE_UNCONSTRAINED_ARTIFACT"
        baseline_spy_fi_df["dag_prefix"] = baseline_spy_fi_df["feature"].map(extract_feature_prefix)
        baseline_spy_fi_df["gain_share"] = baseline_spy_fi_df["gain"] / baseline_spy_fi_df["gain"].sum()
        baseline_spy_fi_df["rank_within_ticker"] = np.arange(1, len(baseline_spy_fi_df) + 1)
        UNCONSTRAINED_IMPORTANCE_DFS.append(
            baseline_spy_fi_df[["ticker", "model_variant", "feature", "dag_prefix", "gain", "gain_share", "rank_within_ticker"]]
        )
        continue

    # =============================================================
    # FIT A FRESH UNCONSTRAINED MODEL WHEN NO SAVED ARTIFACT EXISTS
    # =============================================================

    x_full = EFF_FEATURES_DF.copy()
    y_full = EFF_TARGET_DF[TARGET_COL_MAP[ticker]].copy()
    unconstrained_model = build_xgb_regressor(FROZEN_XGB_PARAMS)
    unconstrained_model.fit(x_full, y_full)
    unconstrained_gain_dict = unconstrained_model.get_booster().get_score(importance_type="gain")
    unconstrained_fi_df = gain_dict_to_dataframe(
        unconstrained_gain_dict,
        ticker=ticker,
        model_variant="BASELINE_UNCONSTRAINED_REFIT",
    )
    UNCONSTRAINED_IMPORTANCE_DFS.append(unconstrained_fi_df)

# ==================================================================
# CONCATENATE UNCONSTRAINED IMPORTANCE TABLES FOR ENTROPY COMPARISON
# ==================================================================

UNCONSTRAINED_FEATURE_IMPORTANCE_DF = pd.concat(UNCONSTRAINED_IMPORTANCE_DFS, ignore_index=True)

# ==============================================================================
# COMPUTE PER-TICKER ENTROPY FOR CONSTRAINED AND UNCONSTRAINED IMPORTANCE TABLES
# ==============================================================================

ENTROPY_ROWS: List[Dict[str, float]] = []

for model_variant, source_df in [
    ("CAUSAL_CONSTRAINED", CAUSAL_FEATURE_IMPORTANCE_DF),
    ("BASELINE_UNCONSTRAINED", UNCONSTRAINED_FEATURE_IMPORTANCE_DF),
]:
    # ==============================================
    # GROUP BY TICKER AND COMPUTE ENTROPY STATISTICS
    # ==============================================

    for ticker, ticker_fi_df in source_df.groupby("ticker", sort=True):
        entropy_stats = compute_entropy_stats(ticker_fi_df)
        ENTROPY_ROWS.append(
            {
                "ticker": ticker,
                "model_variant": model_variant,
                **entropy_stats,
            }
        )

# ======================================================
# BUILD WIDE ENTROPY COMPARISON TABLE WITH DELTA COLUMNS
# ======================================================

ENTROPY_LONG_DF = pd.DataFrame(ENTROPY_ROWS)

ENTROPY_WIDE_DF = (
    ENTROPY_LONG_DF.pivot(
        index="ticker",
        columns="model_variant",
        values=["entropy", "normalized_entropy", "non_zero_feature_count", "top_1_gain_share", "top_5_gain_share"],
    )
    .reset_index()
)

ENTROPY_WIDE_DF.columns = [
    "ticker" if col == ("ticker", "") else f"{col[0]}__{col[1]}"
    for col in ENTROPY_WIDE_DF.columns
]

ENTROPY_WIDE_DF["entropy_delta_causal_minus_unconstrained"] = (
    ENTROPY_WIDE_DF["entropy__CAUSAL_CONSTRAINED"] - ENTROPY_WIDE_DF["entropy__BASELINE_UNCONSTRAINED"]
)
ENTROPY_WIDE_DF["normalized_entropy_delta_causal_minus_unconstrained"] = (
    ENTROPY_WIDE_DF["normalized_entropy__CAUSAL_CONSTRAINED"] - ENTROPY_WIDE_DF["normalized_entropy__BASELINE_UNCONSTRAINED"]
)

# ===========================================================
# SAVE THE ENTROPY COMPARISON TABLE FOR NOTEBOOK 04 REPORTING
# ===========================================================

save_dataframe_csv(ENTROPY_WIDE_DF, NB04_ENTROPY_PATH)

# ===========================================================================
# PRINT THE ENTROPY OUTPUT PATH AND DISPLAY THE TICKER-LEVEL COMPARISON TABLE
# ===========================================================================

print("NB04_ENTROPY_PATH:", str(NB04_ENTROPY_PATH))
display(ENTROPY_WIDE_DF)

***

In [ ]:
# ============
# CODE_BLOCK_J
# ============

# ================================================================
# SELECT A REPRESENTATIVE PLOT TICKER WITH SPY AS THE FIRST CHOICE
# ================================================================

PLOT_TICKER = "SPY" if "SPY" in TICKER_LIST else TICKER_LIST[0]

# ============================================================================
# BUILD A COMBINED LONG-FORM PREDICTION TABLE FOR CAUSAL XGBOOST AND BASELINES
# ============================================================================

PLOT_LONG_PARTS: List[pd.DataFrame] = [CAUSAL_PREDICTIONS_LONG_DF.copy()]

if NB03_BASELINE_PREDICTIONS_DF is not None:
    PLOT_LONG_PARTS.append(NB03_BASELINE_PREDICTIONS_DF.copy())

PLOT_LONG_DF = pd.concat(PLOT_LONG_PARTS, ignore_index=True)

# ============================================================================
# FILTER THE PLOT TABLE TO THE REPRESENTATIVE TICKER FOR FINAL FIGURE CREATION
# ============================================================================

PLOT_TICKER_DF = PLOT_LONG_DF[PLOT_LONG_DF["ticker"] == PLOT_TICKER].copy()
PLOT_TICKER_DF["date"] = pd.to_datetime(PLOT_TICKER_DF["date"])
PLOT_TICKER_DF = PLOT_TICKER_DF.sort_values(["date", "model"]).reset_index(drop=True)

# =========================================================================
# COMPUTE PREFIX-LEVEL GAIN SHARES FOR CONSTRAINED AND UNCONSTRAINED MODELS
# =========================================================================

PREFIX_GAIN_SHARE_DF = (
    pd.concat([CAUSAL_FEATURE_IMPORTANCE_DF, UNCONSTRAINED_FEATURE_IMPORTANCE_DF], ignore_index=True)
    .assign(
        model_family=lambda df: df["model_variant"].map(
            lambda value: "CAUSAL_CONSTRAINED" if "CAUSAL" in value else "BASELINE_UNCONSTRAINED"
        )
    )
    .groupby(["ticker", "model_family", "dag_prefix"], dropna=False)["gain_share"]
    .sum()
    .reset_index()
    .sort_values(["ticker", "model_family", "gain_share"], ascending=[True, True, False])
    .reset_index(drop=True)
)

# =================================================================
# SAVE PREFIX GAIN SHARE TABLE FOR DOWNSTREAM PLOTS AND REPORT TEXT
# =================================================================

save_dataframe_csv(PREFIX_GAIN_SHARE_DF, NB04_PREFIX_GAIN_PATH)

# =====================================================================
# BUILD TOP-FEATURE COMPARISON TABLE FOR THE REPRESENTATIVE PLOT TICKER
# =====================================================================

TOP_FEATURE_COMPARISON_DF = (
    pd.concat([CAUSAL_FEATURE_IMPORTANCE_DF, UNCONSTRAINED_FEATURE_IMPORTANCE_DF], ignore_index=True)
    .assign(
        model_family=lambda df: df["model_variant"].map(
            lambda value: "CAUSAL_CONSTRAINED" if "CAUSAL" in value else "BASELINE_UNCONSTRAINED"
        )
    )
    .query("ticker == @PLOT_TICKER")
    .sort_values(["model_family", "gain"], ascending=[True, False])
    .groupby("model_family", group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

# ==============================================================================
# SAVE THE TOP-FEATURE COMPARISON TABLE FOR WORD INSERTION AND NARRATIVE SUPPORT
# ==============================================================================

save_dataframe_csv(TOP_FEATURE_COMPARISON_DF, NB04_TOP_FEATURE_COMPARISON_PATH)

# ===============================================================================
# PRINT PLOT-TICKER OUTPUT PATHS AND DISPLAY TABLES USED BY THE FINAL FIGURE CELL
# ===============================================================================

print("PLOT_TICKER:", PLOT_TICKER)
print("NB04_PREFIX_GAIN_PATH:", str(NB04_PREFIX_GAIN_PATH))
print("NB04_TOP_FEATURE_COMPARISON_PATH:", str(NB04_TOP_FEATURE_COMPARISON_PATH))
display(PLOT_TICKER_DF.head(20))
display(PREFIX_GAIN_SHARE_DF.head(20))
display(TOP_FEATURE_COMPARISON_DF)

***

In [ ]:
# ============
# CODE_BLOCK_K
# ============

# =============================================================================
# BUILD A PIVOT TABLE OF MODEL PREDICTIONS FOR THE REPRESENTATIVE FORECAST PLOT
# =============================================================================

PLOT_PIVOT_DF = PLOT_TICKER_DF.pivot_table(index="date", columns="model", values="y_pred", aggfunc="first")

# ==============================================================================
# EXTRACT THE REALIZED TARGET SERIES ONCE PER DATE FOR THE REPRESENTATIVE TICKER
# ==============================================================================

REALIZED_PLOT_SERIES = (
    PLOT_TICKER_DF.drop_duplicates(subset=["date"])[["date", "y_true"]]
    .set_index("date")["y_true"]
    .sort_index()
)

# =============================================================
# CREATE RMSE AND MAE DELTA TABLES USED BY THE FINAL BAR CHARTS
# =============================================================

RMSE_DELTA_PLOT_DF = (
    CAUSAL_VS_BASELINE_DF[CAUSAL_VS_BASELINE_DF["ticker"] != "AGGREGATE_EQUAL_WEIGHT"]
    .loc[:, ["ticker", "causal_minus_xgboost_rmse", "causal_minus_ewma_rmse"]]
    .sort_values("causal_minus_xgboost_rmse")
    .reset_index(drop=True)
)

MAE_DELTA_PLOT_DF = (
    CAUSAL_VS_BASELINE_DF[CAUSAL_VS_BASELINE_DF["ticker"] != "AGGREGATE_EQUAL_WEIGHT"]
    .loc[:, ["ticker", "causal_minus_xgboost_mae", "causal_minus_ewma_mae"]]
    .sort_values("causal_minus_xgboost_mae")
    .reset_index(drop=True)
)

# =======================================================================
# BUILD A COMPACT AGGREGATE SUMMARY TABLE FOR NOTEBOOK 04 NARRATIVE CELLS
# =======================================================================

AGGREGATE_SUMMARY_DF = CAUSAL_VS_BASELINE_DF[CAUSAL_VS_BASELINE_DF["ticker"] == "AGGREGATE_EQUAL_WEIGHT"].copy()

# ======================================================
# PRINT SUMMARY STATISTICS AND DISPLAY PLOT-READY TABLES
# ======================================================

print("PLOT_PIVOT_COLUMNS:", list(PLOT_PIVOT_DF.columns))
print("REPRESENTATIVE_REALIZED_OBSERVATIONS:", len(REALIZED_PLOT_SERIES))
display(RMSE_DELTA_PLOT_DF)
display(MAE_DELTA_PLOT_DF)
display(AGGREGATE_SUMMARY_DF)

***

In [ ]:
# ============
# CODE_BLOCK_L
# ============

# ==============================================================
# DEFINE SIMPLE ASSET-CLASS LABELS FOR THE CAPSTONE ETF UNIVERSE
# ==============================================================

ASSET_CLASS_MAP = {
    "SPY": "US_EQUITY",
    "QQQ": "US_EQUITY",
    "IWM": "US_EQUITY",
    "EFA": "INTERNATIONAL_EQUITY",
    "EEM": "INTERNATIONAL_EQUITY",
    "XLF": "SECTOR_EQUITY",
    "XLK": "SECTOR_EQUITY",
    "XLE": "SECTOR_EQUITY",
    "TLT": "FIXED_INCOME",
    "LQD": "FIXED_INCOME",
    "HYG": "FIXED_INCOME",
    "GLD": "COMMODITY",
    "DBC": "COMMODITY",
    "VNQ": "REAL_ESTATE",
}

# =========================================================================
# MERGE ASSET-CLASS LABELS INTO THE CAUSAL VERSUS BASELINE COMPARISON TABLE
# =========================================================================

CAUSAL_VS_BASELINE_WITH_ASSET_CLASS_DF = CAUSAL_VS_BASELINE_DF.copy()
CAUSAL_VS_BASELINE_WITH_ASSET_CLASS_DF["asset_class"] = CAUSAL_VS_BASELINE_WITH_ASSET_CLASS_DF["ticker"].map(ASSET_CLASS_MAP).fillna("UNMAPPED")

# ==========================================================================
# BUILD ASSET-CLASS SUMMARY STATISTICS FOR RMSE AND MAE DELTA INTERPRETATION
# ==========================================================================

ASSET_CLASS_SUMMARY_DF = (
    CAUSAL_VS_BASELINE_WITH_ASSET_CLASS_DF.query('ticker != "AGGREGATE_EQUAL_WEIGHT"')
    .groupby("asset_class", dropna=False)
    .agg(
        ticker_count=("ticker", "count"),
        mean_causal_minus_xgboost_rmse=("causal_minus_xgboost_rmse", "mean"),
        mean_causal_minus_ewma_rmse=("causal_minus_ewma_rmse", "mean"),
        mean_causal_minus_xgboost_mae=("causal_minus_xgboost_mae", "mean"),
        mean_causal_minus_ewma_mae=("causal_minus_ewma_mae", "mean"),
        rmse_wins_vs_xgboost=("rmse_improves_vs_xgboost", "sum"),
        rmse_wins_vs_ewma=("rmse_improves_vs_ewma", "sum"),
    )
    .reset_index()
    .sort_values("mean_causal_minus_xgboost_rmse")
    .reset_index(drop=True)
)

# ==============================================================
# SAVE THE ASSET-CLASS SUMMARY TABLE FOR REPORT SUBSECTION REUSE
# ==============================================================

save_dataframe_csv(ASSET_CLASS_SUMMARY_DF, NB04_ASSET_CLASS_PATH)

# ===============================================================
# PRINT THE ASSET-CLASS OUTPUT PATH AND DISPLAY THE SUMMARY TABLE
# ===============================================================

print("NB04_ASSET_CLASS_PATH:", str(NB04_ASSET_CLASS_PATH))
display(ASSET_CLASS_SUMMARY_DF)

***

In [ ]:
# ============
# CODE_BLOCK_M
# ============

# ========================================================================
# DEFINE THE FINAL NOTEBOOK 04 FIGURE PATHS USED BY THE LAST ARTIFACT CELL
# ========================================================================

NB04_DAG_FIG_PATH = FIGURES_DIR / "notebook04_dag_figure.png"
NB04_FORECAST_FIG_PATH = FIGURES_DIR / f"notebook04_forecast_vs_realized_{PLOT_TICKER}.png"
NB04_RMSE_DELTA_FIG_PATH = FIGURES_DIR / "notebook04_causal_vs_baseline_rmse_delta.png"
NB04_ENTROPY_FIG_PATH = FIGURES_DIR / "notebook04_entropy_comparison.png"
NB04_CAUSAL_IMPORTANCE_FIG_PATH = FIGURES_DIR / f"notebook04_causal_feature_importance_{PLOT_TICKER}.png"
NB04_PREFIX_GAIN_FIG_PATH = FIGURES_DIR / f"notebook04_prefix_gain_share_{PLOT_TICKER}.png"

# ==========================================================================
# ASSEMBLE A PRELIMINARY ARTIFACT REGISTRY TABLE FOR FINAL MANIFEST CREATION
# ==========================================================================

ARTIFACT_REGISTRY_DF = pd.DataFrame(
    [
        {"artifact_type": "TABLE", "artifact_path": str(NB04_CAUSAL_METRICS_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_CAUSAL_PREDICTIONS_TABLE_PATH)},
        {"artifact_type": "DATA", "artifact_path": str(NB04_CAUSAL_PREDICTIONS_DATA_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_GATING_SUMMARY_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_EFFECTIVE_WINDOW_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_SPLIT_SUMMARY_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_COMPARISON_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_FEATURE_IMPORTANCE_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_ENTROPY_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_PREFIX_GAIN_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_TOP_FEATURE_COMPARISON_PATH)},
        {"artifact_type": "TABLE", "artifact_path": str(NB04_ASSET_CLASS_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_DAG_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_FORECAST_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_RMSE_DELTA_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_ENTROPY_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_CAUSAL_IMPORTANCE_FIG_PATH)},
        {"artifact_type": "FIGURE", "artifact_path": str(NB04_PREFIX_GAIN_FIG_PATH)},
    ]
)

# ==========================================================================
# ADD CURRENT FILE-EXISTS FLAGS SO THE FINAL CELL CAN VERIFY FIGURE CREATION
# ==========================================================================

ARTIFACT_REGISTRY_DF["exists_now"] = ARTIFACT_REGISTRY_DF["artifact_path"].map(lambda path: Path(path).exists())

# =======================================================
# PRINT PRELIMINARY REGISTRY STATUS AND DISPLAY THE TABLE
# =======================================================

print("PRELIMINARY_ARTIFACT_COUNT:", len(ARTIFACT_REGISTRY_DF))
display(ARTIFACT_REGISTRY_DF)

***

In [ ]:
# ============
# CODE_BLOCK_N
# ============

# ==========================================================================
# CREATE THE MANUAL DAG FIGURE FOR REPORT INSERTION AND PIPELINE EXPLANATION
# ==========================================================================

fig, ax = plt.subplots(figsize=(12, 4))
ax.set_title("MANUAL DAG FOR NOTEBOOK 04 RISK-STAGE FEATURE GATING")
ax.axis("off")

dag_positions = {
    "SENTIMENT": (0.08, 0.55),
    "MOMENTUM": (0.28, 0.55),
    "RETURNS": (0.48, 0.55),
    "VALUE": (0.28, 0.20),
    "VOLATILITY": (0.48, 0.20),
    "RISK": (0.68, 0.20),
    "ALLOCATION": (0.88, 0.20),
    "MACRO": (0.68, 0.55),
    "REGIME": (0.88, 0.55),
}

for node_name, (x_coord, y_coord) in dag_positions.items():
    # ================================================
    # DRAW A ROUNDED NODE BOX FOR THE CURRENT DAG NODE
    # ================================================

    node_box = FancyBboxPatch(
        (x_coord - 0.07, y_coord - 0.05),
        0.14,
        0.10,
        boxstyle="round,pad=0.02",
        transform=ax.transAxes,
    )
    ax.add_patch(node_box)
    ax.text(x_coord, y_coord, node_name, ha="center", va="center", transform=ax.transAxes)

dag_edges_for_plot = [
    ("SENTIMENT", "MOMENTUM"),
    ("MOMENTUM", "RETURNS"),
    ("VALUE", "RETURNS"),
    ("VOLATILITY", "RISK"),
    ("RISK", "ALLOCATION"),
    ("MACRO", "RISK"),
    ("REGIME", "RISK"),
]

for source_node, target_node in dag_edges_for_plot:
    # ===========================================================
    # DRAW A DIRECTED ARROW FOR THE CURRENT INFORMATION-FLOW EDGE
    # ===========================================================

    x0, y0 = dag_positions[source_node]
    x1, y1 = dag_positions[target_node]
    ax.annotate(
        "",
        xy=(x1 - 0.08, y1),
        xytext=(x0 + 0.08, y0),
        xycoords=ax.transAxes,
        textcoords=ax.transAxes,
        arrowprops={"arrowstyle": "->", "lw": 1.5},
    )

plt.tight_layout()
save_figure_png(fig, NB04_DAG_FIG_PATH, dpi=200)
plt.close(fig)

# ==================================================================
# CREATE FORECAST VERSUS REALIZED PLOT FOR THE REPRESENTATIVE TICKER
# ==================================================================

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(REALIZED_PLOT_SERIES.index, REALIZED_PLOT_SERIES.values, label="REALIZED_FWD_VOL_20D")

for model_name in ["EWMA", "XGBOOST", "CAUSAL_XGBOOST"]:
    # ======================================================
    # PLOT EACH AVAILABLE MODEL SERIES ON THE SAME DATE AXIS
    # ======================================================

    if model_name in PLOT_PIVOT_DF.columns:
        ax.plot(PLOT_PIVOT_DF.index, PLOT_PIVOT_DF[model_name].values, label=model_name)

ax.set_title(f"FORECAST VS REALIZED VOLATILITY FOR {PLOT_TICKER}")
ax.set_xlabel("DATE")
ax.set_ylabel("ANNUALIZED VOLATILITY")
ax.legend()
plt.tight_layout()
save_figure_png(fig, NB04_FORECAST_FIG_PATH, dpi=200)
plt.close(fig)

# ===============================================================
# CREATE RMSE DELTA BAR CHART RELATIVE TO THE TWO BASELINE MODELS
# ===============================================================

fig, ax = plt.subplots(figsize=(14, 5))
rmse_plot_df = RMSE_DELTA_PLOT_DF.set_index("ticker")
rmse_plot_df.plot(kind="bar", ax=ax)
ax.axhline(0.0, linewidth=1.0)
ax.set_title("CAUSAL MODEL RMSE DELTA VERSUS XGBOOST AND EWMA")
ax.set_xlabel("TICKER")
ax.set_ylabel("DELTA RMSE")
plt.xticks(rotation=45)
plt.tight_layout()
save_figure_png(fig, NB04_RMSE_DELTA_FIG_PATH, dpi=200)
plt.close(fig)

# ==================================================================
# CREATE ENTROPY COMPARISON BAR CHART FOR THE REPRESENTATIVE TICKERS
# ==================================================================

fig, ax = plt.subplots(figsize=(12, 5))
entropy_plot_df = ENTROPY_WIDE_DF.set_index("ticker")[
    [
        "normalized_entropy__CAUSAL_CONSTRAINED",
        "normalized_entropy__BASELINE_UNCONSTRAINED",
    ]
]
entropy_plot_df.plot(kind="bar", ax=ax)
ax.set_title("NORMALIZED FEATURE-IMPORTANCE ENTROPY COMPARISON")
ax.set_xlabel("TICKER")
ax.set_ylabel("NORMALIZED ENTROPY")
plt.xticks(rotation=0)
plt.tight_layout()
save_figure_png(fig, NB04_ENTROPY_FIG_PATH, dpi=200)
plt.close(fig)

# ======================================================================
# CREATE TOP-CAUSAL-FEATURE BAR CHART FOR THE REPRESENTATIVE PLOT TICKER
# ======================================================================

fig, ax = plt.subplots(figsize=(12, 6))
causal_top_plot_df = (
    CAUSAL_FEATURE_IMPORTANCE_DF.query("ticker == @PLOT_TICKER")
    .sort_values("gain", ascending=True)
    .tail(15)
)
ax.barh(causal_top_plot_df["feature"], causal_top_plot_df["gain"])
ax.set_title(f"TOP CAUSAL FEATURE IMPORTANCE FOR {PLOT_TICKER}")
ax.set_xlabel("GAIN")
ax.set_ylabel("FEATURE")
plt.tight_layout()
save_figure_png(fig, NB04_CAUSAL_IMPORTANCE_FIG_PATH, dpi=200)
plt.close(fig)

# =====================================================================
# CREATE PREFIX GAIN SHARE BAR CHART FOR THE REPRESENTATIVE PLOT TICKER
# =====================================================================

fig, ax = plt.subplots(figsize=(10, 5))
prefix_plot_df = (
    PREFIX_GAIN_SHARE_DF.query("ticker == @PLOT_TICKER")
    .pivot(index="dag_prefix", columns="model_family", values="gain_share")
    .fillna(0.0)
    .sort_index()
)
prefix_plot_df.plot(kind="bar", ax=ax)
ax.set_title(f"PREFIX GAIN SHARE COMPARISON FOR {PLOT_TICKER}")
ax.set_xlabel("DAG PREFIX")
ax.set_ylabel("TOTAL GAIN SHARE")
plt.xticks(rotation=45)
plt.tight_layout()
save_figure_png(fig, NB04_PREFIX_GAIN_FIG_PATH, dpi=200)
plt.close(fig)

# =========================================================================
# BUILD THE FINAL ARTIFACT MANIFEST AFTER ALL NOTEBOOK 04 FIGURES ARE SAVED
# =========================================================================

NOTEBOOK04_ARTIFACT_MANIFEST_DF = ARTIFACT_REGISTRY_DF.copy()
NOTEBOOK04_ARTIFACT_MANIFEST_DF["exists_now"] = NOTEBOOK04_ARTIFACT_MANIFEST_DF["artifact_path"].map(lambda path: Path(path).exists())
NOTEBOOK04_ARTIFACT_MANIFEST_DF["file_size_bytes"] = NOTEBOOK04_ARTIFACT_MANIFEST_DF["artifact_path"].map(
    lambda path: Path(path).stat().st_size if Path(path).exists() else np.nan
)
save_dataframe_csv(NOTEBOOK04_ARTIFACT_MANIFEST_DF, NB04_ARTIFACT_MANIFEST_PATH)

# ====================================================================
# PRINT FINAL FIGURE PATHS AND DISPLAY THE COMPLETED ARTIFACT MANIFEST
# ====================================================================

print("NB04_DAG_FIG_PATH:", str(NB04_DAG_FIG_PATH))
print("NB04_FORECAST_FIG_PATH:", str(NB04_FORECAST_FIG_PATH))
print("NB04_RMSE_DELTA_FIG_PATH:", str(NB04_RMSE_DELTA_FIG_PATH))
print("NB04_ENTROPY_FIG_PATH:", str(NB04_ENTROPY_FIG_PATH))
print("NB04_CAUSAL_IMPORTANCE_FIG_PATH:", str(NB04_CAUSAL_IMPORTANCE_FIG_PATH))
print("NB04_PREFIX_GAIN_FIG_PATH:", str(NB04_PREFIX_GAIN_FIG_PATH))
print("NB04_ARTIFACT_MANIFEST_PATH:", str(NB04_ARTIFACT_MANIFEST_PATH))
display(NOTEBOOK04_ARTIFACT_MANIFEST_DF)

***